# Analyse exploratoire — Trends Tourisme international en Afrique

## 01 — Introduction et objectifs de l'EDA

Le projet Trends de Gaea21 étudie le tourisme international dans sept destinations : **Afrique du Sud, Égypte, Kenya, Maroc, Maurice, Tanzanie et Tunisie**.

Le dataset maître rassemble les **arrivées**, les **recettes** et la **provenance** des visiteurs. L'EDA doit identifier les données exploitables, préciser les comparaisons défendables et préparer la sélection des indicateurs du futur dashboard.

Questions qui guideront l'EDA :
- Comment les arrivées et les recettes ont-elles évolué ?
- Quelles différences observe-t-on entre destinations ?
- Quelles ruptures temporelles sont visibles ?
- Quelles provenances sont documentées ?
- Quelles comparaisons sont réellement défendables ?

**Périmètre de cette phase :** introduction, chargement, validation, structure et qualité. Les analyses touristiques et les KPI seront développés ultérieurement.

> **Règles méthodologiques**
>
> Valeur manquante ≠ zéro (*missing ≠ zero*). Aucune interpolation, fabrication de valeur ou suppression silencieuse.
> La comparabilité précède la visualisation : unités, métriques, sources et couvertures doivent être compatibles.
> Les pays, régions, totaux, diasporas et institutions restent distincts ; un Top N ou un panel reste partiel.
> Les parts restent en décimal dans les données. Les parts tanzaniennes ne sont pas converties en volumes.
> La provenance égyptienne ne permet aucun classement exhaustif des marchés.

Référentiels : [documentation du projet](../docs/project_documentation.md), [sources](../docs/data_sources.md), [méthodologie](../docs/methodology.md), [dictionnaire](../docs/data_dictionary.md) et [guide de reprise](../docs/handover_guide.md).

## 02 — Chargement et validation du dataset maître

**Objectif :** charger le CSV corrigé depuis la racine du dépôt ou depuis `notebooks/`, puis contrôler son contrat de données. Les fonctions de `src/data_processing.py` sont réutilisées ; la procédure de correction n'est pas exécutée dans l'EDA.

In [1]:
import sys
from pathlib import Path
import hashlib

# Évite de créer des caches dans src lors de cette lecture seule.
sys.dont_write_bytecode = True
import pandas as pd
from IPython.display import display, Markdown

RELATIVE_DATA_PATH = Path("data/final/dataset_maitre_trends_tourisme_afrique.csv")


def find_project_root(start):
    """Cherche le fichier maître depuis le répertoire courant ou son parent."""
    start = Path(start).resolve()
    for candidate in (start, start.parent):
        if (candidate / RELATIVE_DATA_PATH).is_file() and (candidate / "src/data_processing.py").is_file():
            return candidate
    raise FileNotFoundError(
        f"Dataset maître ou module de chargement introuvable depuis {start}. "
        "Ouvrir le notebook depuis la racine du dépôt ou le dossier notebooks."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import (
    load_master_dataset,
    get_dataset_overview,
    get_observations_by_destination,
    get_value_quality_by_layer,
    get_temporal_coverage,
)

In [2]:
DATA_PATH = PROJECT_ROOT / RELATIVE_DATA_PATH
assert DATA_PATH.is_file(), f"Fichier absent : {DATA_PATH}"
source_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()

df = load_master_dataset(DATA_PATH)
df_initial = df.copy(deep=True)  # Témoin d'intégrité, sans transformation.
df.head()

,dataset_layer,destination,iso3,year,origin_name,origin_region,granularity,metric,value,unit,metric_type,coverage_scope,source_name,source_reference,source_file,quality_flag,notes
0,arrivals,Afrique du Sud,ZAF,1995,NaN,NaN,destination_total,tourist_arrivals,4684000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
1,arrivals,Afrique du Sud,ZAF,1996,NaN,NaN,destination_total,tourist_arrivals,5186000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
2,arrivals,Afrique du Sud,ZAF,1997,NaN,NaN,destination_total,tourist_arrivals,5170000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
3,arrivals,Afrique du Sud,ZAF,1998,NaN,NaN,destination_total,tourist_arrivals,5898000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.
4,arrivals,Afrique du Sud,ZAF,1999,NaN,NaN,destination_total,tourist_arrivals,6026000.0,persons,destination_total,Common WDI destination-level series,World Bank WDI,WB_WDI_ST_INT_ARVL,audit_harmonisation_arrivees_7_pays_trends.xlsx,available,No interpolation; missing stays blank.


**Lecture de l'aperçu.** Les premières lignes appartiennent aux arrivées nationales : les champs d'origine y sont sans objet. Cet aperçu ne représente pas toute la diversité des trois couches.

**Validation du schéma.** Le contrat actuel comporte 17 colonnes dans un ordre défini et 1 080 lignes. Toute divergence bloque la suite ; aucune correction automatique n'est appliquée.

In [3]:
EXPECTED_COLUMNS = [
    "dataset_layer", "destination", "iso3", "year", "origin_name",
    "origin_region", "granularity", "metric", "value", "unit",
    "metric_type", "coverage_scope", "source_name", "source_reference",
    "source_file", "quality_flag", "notes",
]


def control_table(checks):
    """Présente les contrôles avant de bloquer sur une éventuelle anomalie."""
    table = pd.DataFrame(checks, columns=["Contrôle", "Résultat attendu", "Résultat observé", "valide"])
    table["Statut"] = table.pop("valide").map({True: "OK", False: "ÉCHEC"})
    display(table)
    assert table["Statut"].eq("OK").all(), "Contrôles échoués : consulter le tableau, sans modifier les données."
    return table


missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
extra_columns = sorted(set(df.columns) - set(EXPECTED_COLUMNS))
schema_controls = control_table([
    ("Nombre de colonnes", "17", str(df.shape[1]), df.shape[1] == 17),
    ("Colonnes manquantes", "Aucune", ", ".join(missing_columns) or "Aucune", not missing_columns),
    ("Colonnes supplémentaires", "Aucune", ", ".join(extra_columns) or "Aucune", not extra_columns),
    ("Ordre des colonnes", "Ordre du dictionnaire", "Conforme" if list(df.columns) == EXPECTED_COLUMNS else "Différent", list(df.columns) == EXPECTED_COLUMNS),
    ("Nombre de lignes", "1080", str(len(df)), len(df) == 1080),
])

,Contrôle,Résultat attendu,Résultat observé,Statut
0,Nombre de colonnes,17,17,OK
1,Colonnes manquantes,Aucune,Aucune,OK
2,Colonnes supplémentaires,Aucune,Aucune,OK
3,Ordre des colonnes,Ordre du dictionnaire,Conforme,OK
4,Nombre de lignes,1080,1080,OK


**Interprétation.** Le schéma correspond à la version corrigée attendue. Cette conformité structurelle ne certifie ni l'exactitude des publications originales ni la comparabilité de toutes les lignes.

**Validation des catégories.** Les modalités ci-dessous reprennent le dictionnaire, y compris `institutional_category` et `missing_unverified`. Les valeurs inattendues, les modalités absentes et les champs catégoriels vides sont signalés.

In [4]:
DESTINATION_ISO = {
    "Afrique du Sud": "ZAF", "Égypte": "EGY", "Kenya": "KEN",
    "Maroc": "MAR", "Maurice": "MUS", "Tanzanie": "TZA", "Tunisie": "TUN",
}
DESTINATIONS = list(DESTINATION_ISO)
LAYERS = ["arrivals", "receipts", "provenance"]
EXPECTED_CATEGORIES = {
    "dataset_layer": set(LAYERS),
    "destination": set(DESTINATIONS),
    "iso3": set(DESTINATION_ISO.values()),
    "granularity": {"destination_total", "aggregate_total", "country", "regional_aggregate", "diaspora", "institutional_category"},
    "metric": {"tourist_arrivals", "tourism_receipts", "tourist_origin"},
    "unit": {"persons", "current_USD", "share"},
    "metric_type": {"destination_total", "tourist_arrivals", "diaspora_arrivals", "source_market_share", "regional_tourist_share", "regional_tourist_nights_share"},
    "quality_flag": {"available", "missing_in_source", "missing_unverified", "exact_aggregate", "exact_country", "exact_diaspora", "exact_top30", "survey_share_top15", "exact_main7", "exact_panel18", "exact_single_country", "regional_share_only"},
}
category_rows = []
for column, expected in EXPECTED_CATEGORIES.items():
    observed = set(df[column].dropna().unique())
    unexpected, absent = observed - expected, expected - observed
    null_count = int(df[column].isna().sum())
    category_rows.append({
        "Variable": column,
        "Modalités documentées": ", ".join(sorted(expected)),
        "Modalités observées": ", ".join(sorted(observed)),
        "Inattendues": ", ".join(sorted(unexpected)) or "Aucune",
        "Documentées absentes": ", ".join(sorted(absent)) or "Aucune",
        "Valeurs manquantes": null_count,
        "Statut": "OK" if not unexpected and not absent and null_count == 0 else "ÉCHEC",
    })
category_controls = pd.DataFrame(category_rows)
display(category_controls)
assert category_controls["Statut"].eq("OK").all(), "Modalités différentes du dictionnaire : investigation nécessaire."
assert df["iso3"].eq(df["destination"].map(DESTINATION_ISO)).all(), "Couple destination/ISO3 incohérent."

,Variable,Modalités documentées,Modalités observées,Inattendues,Documentées absentes,Valeurs manquantes,Statut
0,dataset_layer,"arrivals, provenance, receipts","arrivals, provenance, receipts",Aucune,Aucune,0,OK
1,destination,"Afrique du Sud, Kenya, Maroc, Maurice, Tanzani...","Afrique du Sud, Kenya, Maroc, Maurice, Tanzani...",Aucune,Aucune,0,OK
2,iso3,"EGY, KEN, MAR, MUS, TUN, TZA, ZAF","EGY, KEN, MAR, MUS, TUN, TZA, ZAF",Aucune,Aucune,0,OK
3,granularity,"aggregate_total, country, destination_total, d...","aggregate_total, country, destination_total, d...",Aucune,Aucune,0,OK
4,metric,"tourism_receipts, tourist_arrivals, tourist_or...","tourism_receipts, tourist_arrivals, tourist_or...",Aucune,Aucune,0,OK
5,unit,"current_USD, persons, share","current_USD, persons, share",Aucune,Aucune,0,OK
6,metric_type,"destination_total, diaspora_arrivals, regional...","destination_total, diaspora_arrivals, regional...",Aucune,Aucune,0,OK
7,quality_flag,"available, exact_aggregate, exact_country, exa...","available, exact_aggregate, exact_country, exa...",Aucune,Aucune,0,OK


**Interprétation.** Aucune modalité inattendue n'apparaît. `country` reste une catégorie du modèle : Réunion y est conservée comme marché distinct et ne doit pas être fusionnée avec France. L'institution ONU et les groupes régionaux doivent rester séparés des pays.

**Cohérences et corrections approuvées.** Contrôler les types numériques, les unités des couches nationales, les parts décimales et les trois corrections. Les champs de traçabilité doivent être renseignés ; leur présence n'est pas une vérification de source.

In [5]:
national_mask = df["dataset_layer"].isin(["arrivals", "receipts"])
provenance_mask = df["dataset_layer"].eq("provenance")
scand = df.loc[df["destination"].eq("Tunisie") & df["origin_name"].eq("Scandinaves")]
uno = df.loc[df["destination"].eq("Kenya") & df["origin_name"].eq("United Nations Organization")]
tun_missing = df.loc[provenance_mask & df["destination"].eq("Tunisie") & df["value"].isna() & df["year"].isin([2017, 2018])]
trace_columns = ["source_name", "source_reference", "coverage_scope", "notes"]
expected_metric = df["dataset_layer"].map({"arrivals": "tourist_arrivals", "receipts": "tourism_receipts", "provenance": "tourist_origin"})
national_units = df.loc[national_mask, "dataset_layer"].map({"arrivals": "persons", "receipts": "current_USD"})
share_values = df.loc[df["unit"].eq("share"), "value"].dropna()

integrity_controls = control_table([
    ("Année entière sans absence", "Vrai", str(pd.api.types.is_integer_dtype(df["year"]) and df["year"].notna().all()), pd.api.types.is_integer_dtype(df["year"]) and df["year"].notna().all()),
    ("Value numérique", "Vrai", str(pd.api.types.is_numeric_dtype(df["value"])), pd.api.types.is_numeric_dtype(df["value"])),
    ("Metric cohérente avec la couche", "Vrai", str(df["metric"].eq(expected_metric).all()), df["metric"].eq(expected_metric).all()),
    ("Unités nationales", "persons / current_USD", ", ".join(sorted(df.loc[national_mask, "unit"].unique())), df.loc[national_mask, "unit"].eq(national_units).all()),
    ("Totaux nationaux", "destination_total", ", ".join(df.loc[national_mask, "granularity"].unique()), df.loc[national_mask, ["granularity", "metric_type"]].eq("destination_total").all().all()),
    ("Parts décimales", "Entre 0 et 1", f"{share_values.min():.3f} à {share_values.max():.3f}", share_values.between(0, 1).all()),
    ("Traçabilité renseignée", "0 champ vide", str(int(df[trace_columns].isna().sum().sum())), df[trace_columns].notna().all().all() and df[trace_columns].apply(lambda s: s.str.strip().ne("").all()).all()),
    ("Scandinaves", "7 lignes, 2017–2023, agrégat", str(len(scand)), len(scand) == 7 and set(scand["year"]) == set(range(2017, 2024)) and scand["granularity"].eq("regional_aggregate").all() and scand["quality_flag"].eq("exact_aggregate").all()),
    ("ONU Kenya", "1 ligne 2022, institution, exact_top30", str(len(uno)), len(uno) == 1 and uno["year"].eq(2022).all() and uno["granularity"].eq("institutional_category").all() and uno["quality_flag"].eq("exact_top30").all()),
    ("Absences tunisiennes", "60 lignes missing_unverified", str(len(tun_missing)), len(tun_missing) == 60 and tun_missing["quality_flag"].eq("missing_unverified").all() and tun_missing["value"].isna().all()),
])

,Contrôle,Résultat attendu,Résultat observé,Statut
0,Année entière sans absence,Vrai,True,OK
1,Value numérique,Vrai,True,OK
2,Metric cohérente avec la couche,Vrai,True,OK
3,Unités nationales,persons / current_USD,"current_USD, persons",OK
4,Totaux nationaux,destination_total,destination_total,OK
5,Parts décimales,Entre 0 et 1,0.016 à 0.643,OK
6,Traçabilité renseignée,0 champ vide,0,OK
7,Scandinaves,"7 lignes, 2017–2023, agrégat",7,OK
8,ONU Kenya,"1 ligne 2022, institution, exact_top30",1,OK
9,Absences tunisiennes,60 lignes missing_unverified,60,OK


**Interprétation.** Les trois corrections approuvées sont présentes. Les parts restent en décimal et les couches nationales gardent leurs unités distinctes. Les causes des absences tunisiennes et certaines références originales restent à vérifier.

**Clés et doublons.** Les clés logiques suivent le dictionnaire : couche–destination–année–métrique–type pour les séries nationales ; couche–destination–année–origine–granularité–type–unité pour la provenance. On compte les lignes en excès avec `keep="first"` et les lignes impliquées avec `keep=False`. Aucune ligne n'est supprimée.

In [6]:
NATIONAL_KEY = ["dataset_layer", "destination", "year", "metric", "metric_type"]
PROVENANCE_KEY = ["dataset_layer", "destination", "year", "origin_name", "granularity", "metric_type", "unit"]
duplicate_checks = [
    ("Complets", df, None),
    ("Logiques — séries nationales", df.loc[national_mask], NATIONAL_KEY),
    ("Logiques — provenance", df.loc[provenance_mask], PROVENANCE_KEY),
]
duplicate_rows = []
for label, subset, key in duplicate_checks:
    excess = int(subset.duplicated(subset=key).sum())
    involved = subset.duplicated(subset=key, keep=False)
    duplicate_rows.append({
        "Contrôle": label, "Lignes en excès": excess,
        "Lignes impliquées": int(involved.sum()),
        "Clés incomplètes": int(subset[key].isna().any(axis=1).sum()) if key else 0,
    })
    if involved.any():
        display(subset.loc[involved])
duplicate_summary = pd.DataFrame(duplicate_rows)
display(duplicate_summary)
assert duplicate_summary[["Lignes en excès", "Clés incomplètes"]].eq(0).all().all(), "Doublons ou clés incomplètes : aucune suppression automatique."

,Contrôle,Lignes en excès,Lignes impliquées,Clés incomplètes
0,Complets,0,0,0
1,Logiques — séries nationales,0,0,0
2,Logiques — provenance,0,0,0


**Interprétation.** Aucun doublon complet ou logique n'est détecté et les clés sont renseignées. Des lignes de même destination et année appartenant à des couches ou origines différentes sont légitimes.

## 03 — Structure et qualité des données

Avant l'analyse statistique, distinguer les lignes du dataset, les valeurs renseignées et les périmètres réellement comparables. Les contrôles ci-dessous ne modifient aucune donnée.

### 3.1 Dimensions du dataset

**Question :** quelle est la taille du dataset et quel périmètre général représente-t-il ?

In [7]:
overview = get_dataset_overview(df)
dimensions = pd.DataFrame({
    "Mesure": ["Lignes", "Colonnes", "Destinations", "Couches", "Première année (lignes)", "Dernière année (lignes)"],
    "Résultat": [overview["shape"][0], overview["shape"][1], df["destination"].nunique(), df["dataset_layer"].nunique(), df["year"].min(), df["year"].max()],
})
display(dimensions)
display(pd.DataFrame({"Destination": DESTINATIONS, "ISO3": [DESTINATION_ISO[d] for d in DESTINATIONS]}))

,Mesure,Résultat
0,Lignes,1080
1,Colonnes,17
2,Destinations,7
3,Couches,3
4,Première année (lignes),1995
5,Dernière année (lignes),2024


,Destination,ISO3
0,Afrique du Sud,ZAF
1,Égypte,EGY
2,Kenya,KEN
3,Maroc,MAR
4,Maurice,MUS
5,Tanzanie,TZA
6,Tunisie,TUN


**Interprétation.** Le dataset réunit 1 080 lignes, 17 colonnes, sept destinations et trois couches sur 1995–2024. Cette période globale n'est pas celle de chaque série. Le nombre de lignes par destination ne mesure pas son importance touristique.

### 3.2 Types de variables

**Question :** quels types pandas et quels rôles analytiques portent les 17 variables ?

In [8]:
roles = [
    "Couche analytique", "Destination étudiée", "Code géographique de destination",
    "Année de référence", "Libellé source du marché ou groupe", "Région selon la source",
    "Niveau statistique", "Famille d'indicateur", "Valeur à lire avec son contexte",
    "Unité de mesure", "Nature précise de la mesure", "Périmètre réellement couvert",
    "Producteur ou source déclarée", "Référence documentaire", "Fichier de la chaîne de collecte",
    "Disponibilité / qualité / couverture", "Précautions et limites",
]
variable_types = pd.DataFrame({
    "variable": EXPECTED_COLUMNS,
    "dtype": [str(df[c].dtype) for c in EXPECTED_COLUMNS],
    "rôle": roles,
})
display(variable_types)

,variable,dtype,rôle
0,dataset_layer,object,Couche analytique
1,destination,object,Destination étudiée
2,iso3,object,Code géographique de destination
3,year,int64,Année de référence
4,origin_name,object,Libellé source du marché ou groupe
5,origin_region,object,Région selon la source
6,granularity,object,Niveau statistique
7,metric,object,Famille d'indicateur
8,value,float64,Valeur à lire avec son contexte
9,unit,object,Unité de mesure


**Interprétation.** `year` est entier et `value` numérique ; les autres champs portent le contexte. Les valeurs de `value` ne peuvent pas être additionnées globalement : personnes, USD courants et parts ne mesurent pas la même chose.

### 3.3 Valeurs manquantes

**Questions :** quelles colonnes sont incomplètes, et où les valeurs numériques manquent-elles ? Séparer les champs d'origine sans objet, les absences confirmées dans la source et les causes non vérifiées.

In [9]:
missing_by_column = pd.DataFrame({
    "Variable": df.columns,
    "Valeurs manquantes": df.isna().sum().to_numpy(),
    "Part des lignes (%)": (df.isna().mean() * 100).round(2).to_numpy(),
})
display(Markdown("**A — Absences par colonne**"))
display(missing_by_column)
display(Markdown("**B — Valeurs numériques par couche**"))
controle_valeurs = get_value_quality_by_layer(df).rename(columns={
    "observations": "lignes", "valeurs_disponibles": "valeurs renseignées", "valeurs_manquantes": "valeurs manquantes",
}).reindex(LAYERS)
display(controle_valeurs)
display(Markdown("**C — Valeurs numériques par destination**"))
missing_by_destination = df.groupby("destination")["value"].agg(
    lignes="size", renseignees="count", manquantes=lambda s: s.isna().sum(),
).reindex(DESTINATIONS).rename(columns={"renseignees": "valeurs renseignées", "manquantes": "valeurs manquantes"})
display(missing_by_destination)

**A — Absences par colonne**

,Variable,Valeurs manquantes,Part des lignes (%)
0,dataset_layer,0,0.0
1,destination,0,0.0
2,iso3,0,0.0
3,year,0,0.0
4,origin_name,364,33.7
5,origin_region,364,33.7
6,granularity,0,0.0
7,metric,0,0.0
8,value,67,6.2
9,unit,0,0.0


**B — Valeurs numériques par couche**

,lignes,valeurs renseignées,valeurs manquantes
dataset_layer,,,
arrivals,182,179,3
receipts,182,178,4
provenance,716,656,60


**C — Valeurs numériques par destination**

,lignes,valeurs renseignées,valeurs manquantes
destination,,,
Afrique du Sud,106,106,0
Égypte,70,69,1
Kenya,142,140,2
Maroc,169,169,0
Maurice,73,73,0
Tanzanie,97,93,4
Tunisie,423,363,60


In [10]:
display(Markdown("**D — Nature des absences et cas particuliers**"))
structural_missing = df.groupby("dataset_layer")[["origin_name", "origin_region"]].agg(lambda s: s.isna().sum()).reindex(LAYERS)
display(structural_missing.rename(columns={"origin_name": "origin_name absent", "origin_region": "origin_region absent"}))

missing_values = df.loc[df["value"].isna()]
missing_reasons = pd.DataFrame({
    "Nature": ["missing_in_source", "missing_unverified", "Autres valeurs numériques manquantes"],
    "Lignes": [
        int(missing_values["quality_flag"].eq("missing_in_source").sum()),
        int(missing_values["quality_flag"].eq("missing_unverified").sum()),
        int((~missing_values["quality_flag"].isin(["missing_in_source", "missing_unverified"])).sum()),
    ],
})
display(missing_reasons)
display(missing_values.groupby(["dataset_layer", "destination", "year", "quality_flag"], dropna=False).size().rename("valeurs manquantes").to_frame())
missing_flags = df["quality_flag"].isin(["missing_in_source", "missing_unverified"])
assert df["value"].isna().eq(missing_flags).all(), "Incohérence entre disponibilité et quality_flag."
assert df.loc[national_mask, ["origin_name", "origin_region"]].isna().all().all()
assert df.loc[provenance_mask, ["origin_name", "origin_region"]].notna().all().all()

**D — Nature des absences et cas particuliers**

,origin_name absent,origin_region absent
dataset_layer,,
arrivals,182,182
receipts,182,182
provenance,0,0


,Nature,Lignes
0,missing_in_source,7
1,missing_unverified,60
2,Autres valeurs numériques manquantes,0


valeurs manquantes
dataset_layer destination year quality_flag                          
arrivals      Kenya       2020 missing_in_source                    1
              Tanzanie    2020 missing_in_source                    1
              Égypte      2020 missing_in_source                    1
provenance    Tunisie     2017 missing_unverified                  30
                          2018 missing_unverified                  30
receipts      Kenya       2020 missing_in_source                    1
              Tanzanie    1995 missing_in_source                    1
                          1996 missing_in_source                    1
                          2020 missing_in_source                    1

**Interprétation.** Les 364 absences de chacun des champs d'origine sont structurelles et concernent les totaux nationaux. Les 67 valeurs numériques absentes se répartissent entre 7 `missing_in_source` et 60 `missing_unverified` tunisiennes en 2017–2018 ; aucune autre absence numérique n'apparaît. La cause des 60 absences tunisiennes reste inconnue : ni zéro, ni absence confirmée dans la source. Aucune interpolation n'est réalisée.

### 3.4 Doublons

**Question :** les contrôles de clés permettent-ils de poursuivre sans dédoublonnage ? Le résultat de la section 02 est réutilisé, sans recalcul ni suppression.

In [11]:
display(duplicate_summary)

,Contrôle,Lignes en excès,Lignes impliquées,Clés incomplètes
0,Complets,0,0,0
1,Logiques — séries nationales,0,0,0
2,Logiques — provenance,0,0,0


**Interprétation.** Aucun doublon n'a été détecté selon les clés retenues. Il n'y a donc aucun dédoublonnage à effectuer ; ce contrôle ne remplace pas la vérification des sources.

### 3.5 Couverture par dimension

**Question :** combien de lignes, de valeurs renseignées et de valeurs manquantes chaque destination possède-t-elle dans chaque couche ?

In [12]:
# La fonction existante compte les lignes, y compris celles sans value.
observations_par_pays = get_observations_by_destination(df).reindex(index=DESTINATIONS, columns=LAYERS)
pair_quality = df.groupby(["destination", "dataset_layer"])["value"].agg(
    lignes="size", renseignees="count", manquantes=lambda s: s.isna().sum(),
).rename(columns={"renseignees": "valeurs renseignées", "manquantes": "valeurs manquantes"})
assert observations_par_pays.equals(pair_quality["lignes"].unstack("dataset_layer").reindex(index=DESTINATIONS, columns=LAYERS))
coverage_by_dimension = pair_quality.unstack("dataset_layer").swaplevel(0, 1, axis=1)
coverage_by_dimension = coverage_by_dimension.reindex(index=DESTINATIONS, columns=pd.MultiIndex.from_product(
    [LAYERS, ["lignes", "valeurs renseignées", "valeurs manquantes"]],
    names=["dataset_layer", "mesure"],
))
display(coverage_by_dimension)

dataset_layer  arrivals                                        receipts  \
mesure           lignes valeurs renseignées valeurs manquantes   lignes   
destination                                                               
Afrique du Sud       26                  26                  0       26   
Égypte               26                  25                  1       26   
Kenya                26                  25                  1       26   
Maroc                26                  26                  0       26   
Maurice              26                  26                  0       26   
Tanzanie             26                  25                  1       26   
Tunisie              26                  26                  0       26   

dataset_layer                                         provenance  \
mesure         valeurs renseignées valeurs manquantes     lignes   
destination                                                        
Afrique du Sud                  26                  0         54   
Égypte                          26                  0         18   
Kenya                           25                  1         90   
Maroc                           26                  0        117   
Maurice                         26                  0         21   
Tanzanie                        23                  3         45   
Tunisie                         26                  0        371   

dataset_layer                                          
mesure         valeurs renseignées valeurs manquantes  
destination                                            
Afrique du Sud                  54                  0  
Égypte                          18                  0  
Kenya                           90                  0  
Maroc                          117                  0  
Maurice                         21                  0  
Tanzanie                        45                  0  
Tunisie                        311                 60

**Lecture du tableau.** Les couches nationales ont chacune 26 lignes par destination, mais certaines valeurs manquent. La provenance varie selon les périodes, les catégories et les panels publiés. Ces effectifs ne permettent ni de comparer les volumes touristiques ni de conclure à une couverture exhaustive.

**Contexte de couverture.** Identifier explicitement unités, granularités, métriques, sources et panels avant toute comparaison. Le tableau suivant inventorie des périmètres, sans sommer les valeurs touristiques.

In [13]:
context_columns = ["destination", "unit", "granularity", "metric_type", "coverage_scope", "quality_flag"]
provenance_context = df.loc[provenance_mask].groupby(context_columns, dropna=False, sort=True).agg(
    lignes=("value", "size"), renseignees=("value", "count"),
).rename(columns={"renseignees": "valeurs renseignées"}).reset_index()
display(provenance_context)
traceability_summary = df.groupby(["destination", "dataset_layer"]).agg(
    sources=("source_name", lambda s: " ; ".join(sorted(s.unique()))),
    references=("source_reference", "nunique"),
    notes_distinctes=("notes", "nunique"),
)
display(traceability_summary)

,destination,unit,granularity,metric_type,coverage_scope,quality_flag,lignes,valeurs renseignées
0,Afrique du Sud,persons,country,tourist_arrivals,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18,54,54
1,Kenya,persons,country,tourist_arrivals,Top 30 marchés sources publiés,exact_top30,89,89
2,Kenya,persons,institutional_category,tourist_arrivals,Top 30 marchés sources publiés,exact_top30,1,1
3,Maroc,persons,aggregate_total,tourist_arrivals,Série officielle incluant agrégats publiés,exact_aggregate,9,9
4,Maroc,persons,country,tourist_arrivals,Nationalités/pays publiés par Open Data Maroc,exact_country,81,81
5,Maroc,persons,diaspora,diaspora_arrivals,MRE séparés des touristes étrangers,exact_diaspora,9,9
6,Maroc,persons,regional_aggregate,tourist_arrivals,Série officielle incluant agrégats publiés,exact_aggregate,18,18
7,Maurice,persons,country,tourist_arrivals,7 principaux marchés publiés dans les annual h...,exact_main7,21,21
8,Tanzanie,share,country,source_market_share,Top 15 marchés — parts issues de l'Exit Survey,survey_share_top15,45,45
9,Tunisie,persons,country,tourist_arrivals,Nationalités publiées par ONTT,exact_country,297,297


sources  \
destination    dataset_layer                                                      
Afrique du Sud arrivals                                          World Bank WDI   
               provenance                               Statistics South Africa   
               receipts                                          World Bank WDI   
Kenya          arrivals                                          World Bank WDI   
               provenance             TRI / Directorate of Immigration Services   
               receipts                                          World Bank WDI   
Maroc          arrivals                                          World Bank WDI   
               provenance        Open Data Maroc / Ministère chargé du Tourisme   
               receipts                                          World Bank WDI   
Maurice        arrivals                                          World Bank WDI   
               provenance                                  Statistics Mauritius   
               receipts                                          World Bank WDI   
Tanzanie       arrivals                                          World Bank WDI   
               provenance     Tanzania NBS / International Visitors' Exit Su...   
               receipts                                          World Bank WDI   
Tunisie        arrivals                                          World Bank WDI   
               provenance                                                  ONTT   
               receipts                                          World Bank WDI   
Égypte         arrivals                                          World Bank WDI   
               provenance     CAPMAS Statistical Yearbook - Tourism ; User-p...   
               receipts                                          World Bank WDI   

                              references  notes_distinctes  
destination    dataset_layer                                
Afrique du Sud arrivals                1                 1  
               provenance              3                18  
               receipts                1                 1  
Kenya          arrivals                1                 1  
               provenance              3                31  
               receipts                1                 1  
Maroc          arrivals                1                 1  
               provenance              1                 1  
               receipts                1                 1  
Maurice        arrivals                1                 1  
               provenance              3                 7  
               receipts                1                 1  
Tanzanie       arrivals                1                 1  
               provenance              3                15  
               receipts                1                 1  
Tunisie        arrivals                1                 1  
               provenance              1                 3  
               receipts                1                 1  
Égypte         arrivals                1                 1  
               provenance              2                 3  
               receipts                1                 1

**Interprétation.** Kenya : Top 30 avec une catégorie institutionnelle ; Tanzanie : parts du Top 15 ; Maurice : panel de sept marchés ; Afrique du Sud : panel de 18 marchés. Maroc et Tunisie conservent des groupes et diasporas distincts. Égypte : un seul marché pays et deux métriques régionales séparées, sans classement exhaustif. Les références présentes ne prouvent pas leur vérification : la liaison CAPMAS aux dix observations États-Unis reste non vérifiable avec les pièces du dépôt. Les libellés multilingues ne sont pas harmonisés ici.

### 3.6 Couverture temporelle par destination

**Question :** les années présentes dans les lignes correspondent-elles à des valeurs renseignées ? Calculer séparément les deux couvertures, sans créer d'année ni de valeur.

In [14]:
# La fonction existante donne la couverture des lignes ; on la complète localement.
couverture_temporelle = get_temporal_coverage(df).rename(columns={
    "annee_debut": "année première ligne", "annee_fin": "année dernière ligne", "observations": "lignes",
}).set_index(["destination", "dataset_layer"])
available_df = df.loc[df["value"].notna()]
valid_coverage = available_df.groupby(["destination", "dataset_layer"]).agg(
    premiere=("year", "min"), derniere=("year", "max"),
    annees=("year", "nunique"),
).rename(columns={"premiere": "première année renseignée", "derniere": "dernière année renseignée", "annees": "années avec valeur"})
couverture_temporelle = couverture_temporelle.join(valid_coverage).join(
    pair_quality[["valeurs renseignées", "valeurs manquantes"]]
)
couverture_temporelle = couverture_temporelle.reindex(pd.MultiIndex.from_product(
    [DESTINATIONS, LAYERS], names=["destination", "dataset_layer"],
))
display(couverture_temporelle[[
    "année première ligne", "année dernière ligne", "première année renseignée",
    "dernière année renseignée", "lignes", "valeurs renseignées",
    "valeurs manquantes", "années avec valeur",
]])

année première ligne  année dernière ligne  \
destination    dataset_layer                                               
Afrique du Sud arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Égypte         arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2010                  2019   
Kenya          arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Maroc          arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2012                  2020   
Maurice        arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Tanzanie       arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2022                  2024   
Tunisie        arrivals                       1995                  2020   
               receipts                       1995                  2020   
               provenance                     2017                  2023   

                              première année renseignée  \
destination    dataset_layer                              
Afrique du Sud arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Égypte         arrivals                            1995   
               receipts                            1995   
               provenance                          2010   
Kenya          arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Maroc          arrivals                            1995   
               receipts                            1995   
               provenance                          2012   
Maurice        arrivals                            1995   
               receipts                            1995   
               provenance                          2022   
Tanzanie       arrivals                            1995   
               receipts                            1997   
               provenance                          2022   
Tunisie        arrivals                            1995   
               receipts                            1995   
               provenance                          2017   

                              dernière année renseignée  lignes  \
destination    dataset_layer                                      
Afrique du Sud arrivals                            2020      26   
               receipts                            2020      26   
               provenance                          2024      54   
Égypte         arrivals                            2019      26   
               receipts                            2020      26   
               provenance                          2019      18   
Kenya          arrivals                            2019      26   
               receipts                            2019      26   
               provenance                          2024      90   
Maroc          arrivals                            2020      26   
               receipts                            2020      26   
               provenance                          2

**Lecture de la couverture temporelle.** Les bornes des lignes surestiment parfois la disponibilité : une ligne peut exister sans valeur. Pour la provenance, une année renseignée signifie au moins une valeur, pas une couverture complète des marchés. La composition et le périmètre restent propres à chaque source.

In [15]:
def common_years(layer):
    """Intersection des années avec au moins une valeur pour chacune des 7 destinations."""
    selected = available_df.loc[available_df["dataset_layer"].eq(layer)]
    per_destination = [
        set(selected.loc[selected["destination"].eq(destination), "year"])
        for destination in DESTINATIONS
    ]
    return set.intersection(*per_destination)


common_by_layer = {layer: common_years(layer) for layer in LAYERS}
common_national = common_by_layer["arrivals"] & common_by_layer["receipts"]
last_common_national = max(common_national) if common_national else None
display(pd.DataFrame([
    {"Couche": layer, "Années communes renseignées": ", ".join(map(str, sorted(years))) or "Aucune",
     "Dernière année commune": max(years) if years else pd.NA}
    for layer, years in common_by_layer.items()
]))
post_2020 = available_df.loc[available_df["year"].gt(2020) & available_df["dataset_layer"].isin(["arrivals", "receipts"])]
display(pd.DataFrame({
    "Contrôle": ["Dernière année commune aux deux séries nationales", "Valeurs nationales après 2020", "Années communes de provenance"],
    "Résultat": [str(last_common_national) if last_common_national else "Aucune", str(len(post_2020)), str(len(common_by_layer["provenance"]))],
}))
recovery_statement = (
    "Les séries nationales ne permettent donc pas actuellement d'étudier une reprise post-Covid."
    if post_2020.empty else
    "Des valeurs nationales après 2020 existent ; leur couverture doit être évaluée avant toute étude de reprise."
)
origin_statement = (
    "Il n'existe aucune année commune renseignée de provenance aux sept destinations."
    if not common_by_layer["provenance"] else
    "Des années de provenance sont communes, sans garantie de comparabilité des périmètres."
)
display(Markdown(
    f"**Constat recalculé.** Dernière année commune aux deux séries nationales : **{last_common_national}**. "
    f"{recovery_statement} {origin_statement}"
))

,Couche,Années communes renseignées,Dernière année commune
0,arrivals,"1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002...",2019
1,receipts,"1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004...",2019
2,provenance,Aucune,<NA>


,Contrôle,Résultat
0,Dernière année commune aux deux séries nationales,2019
1,Valeurs nationales après 2020,0
2,Années communes de provenance,0


**Constat recalculé.** Dernière année commune aux deux séries nationales : **2019**. Les séries nationales ne permettent donc pas actuellement d'étudier une reprise post-Covid. Il n'existe aucune année commune renseignée de provenance aux sept destinations.

**Interprétation.** L'étendue globale jusqu'en 2024 vient de certaines provenances et ne prolonge pas les séries nationales. Une intersection d'années est une condition nécessaire, mais insuffisante, à la comparabilité. Les couvertures partielles ne doivent pas servir à reconstruire des totaux nationaux.

In [16]:
# Vérification finale : aucune mutation du DataFrame maître ni écriture du CSV.
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash, "Le fichier maître a changé."
print("Intégrité validée : données en mémoire et fichier maître inchangés.")

Intégrité validée : données en mémoire et fichier maître inchangés.


**Fin de la phase 1.** Le schéma, les catégories, les clés et la couverture ont été contrôlés. Les réserves de source et de périmètre restent applicables. Aucune analyse de tendance ni aucun KPI n'est développé dans cette phase.

## 04 — Analyse des arrivées touristiques

**Objectif :** décrire les trajectoires des sept destinations à partir de la seule couche `arrivals`, déjà chargée dans `df`. Les comparaisons utilisent des périodes explicites et les calculs ne portent que sur des valeurs renseignées. La provenance ne sert jamais à reconstruire des totaux nationaux.

**Préparation et contrôle.** Conserver les métadonnées de source, unité, couverture et qualité. Toute incohérence est signalée par une assertion ; aucune correction automatique n'est effectuée.

In [17]:
import altair as alt
import numpy as np

arrivals_df = df.loc[df["dataset_layer"].eq("arrivals")].copy()
arrivals_initial = arrivals_df.copy(deep=True)
arrivals_available = arrivals_df.loc[arrivals_df["value"].notna()].copy()
assert arrivals_df["dataset_layer"].eq("arrivals").all()
assert set(arrivals_df["destination"]) == set(DESTINATIONS)
assert not arrivals_df.duplicated(["destination", "year"]).any()
for column, expected in {
    "unit": "persons", "metric": "tourist_arrivals",
    "metric_type": "destination_total", "granularity": "destination_total",
}.items():
    assert arrivals_df[column].eq(expected).all(), f"Arrivals : {column} incohérent."
assert set(arrivals_df["quality_flag"]) <= {"available", "missing_in_source"}
assert arrivals_df["value"].isna().eq(arrivals_df["quality_flag"].eq("missing_in_source")).all()
assert np.isfinite(arrivals_available["value"]).all()
assert arrivals_available["value"].ge(0).all(), "Valeur négative : vérifier la source."
arrivals_coverage = arrivals_df.groupby("destination").agg(
    premiere_ligne=("year", "min"), derniere_ligne=("year", "max"),
    lignes=("value", "size"), valeurs_renseignees=("value", "count"),
    valeurs_manquantes=("value", lambda s: s.isna().sum()),
).join(arrivals_available.groupby("destination").agg(
    premiere_annee_renseignee=("year", "min"), derniere_annee_renseignee=("year", "max"),
)).reindex(DESTINATIONS)
display(arrivals_coverage)
display(arrivals_df[["unit", "metric", "metric_type", "granularity", "quality_flag", "source_name", "source_reference", "coverage_scope"]].drop_duplicates())

# Formats réservés à l'affichage ; les DataFrames restent numériques.
def format_count(value):
    return f"{value:,.0f}".replace(",", " ") if pd.notna(value) else "Non renseigné"


def format_percent(value):
    return f"{value:+.2f} %".replace(".", ",") if pd.notna(value) else "Non calculable"


def show_arrivals_table(table, counts=(), percentages=()):
    formats = {column: format_count for column in counts}
    formats.update({column: format_percent for column in percentages})
    display(table.style.format(formats, na_rep="Non renseigné"))


arrival_colors = alt.Scale(domain=DESTINATIONS, range=[
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#a6761d", "#1f78b4",
])
arrival_source_note = "Source : World Bank WDI — ST.INT.ARVL ; valeurs renseignées uniquement."
arrival_charts = []

,premiere_ligne,derniere_ligne,lignes,valeurs_renseignees,valeurs_manquantes,premiere_annee_renseignee,derniere_annee_renseignee
destination,,,,,,,
Afrique du Sud,1995,2020,26,26,0,1995,2020
Égypte,1995,2020,26,25,1,1995,2019
Kenya,1995,2020,26,25,1,1995,2019
Maroc,1995,2020,26,26,0,1995,2020
Maurice,1995,2020,26,26,0,1995,2020
Tanzanie,1995,2020,26,25,1,1995,2019
Tunisie,1995,2020,26,26,0,1995,2020


,unit,metric,metric_type,granularity,quality_flag,source_name,source_reference,coverage_scope
0,persons,tourist_arrivals,destination_total,destination_total,available,World Bank WDI,WB_WDI_ST_INT_ARVL,Common WDI destination-level series
51,persons,tourist_arrivals,destination_total,destination_total,missing_in_source,World Bank WDI,WB_WDI_ST_INT_ARVL,Common WDI destination-level series


**Interprétation.** La couche contient 182 lignes et 179 valeurs renseignées. Les séries commencent en 1995 ; les dernières valeurs sont en 2020 pour quatre destinations, en 2019 pour l'Égypte, le Kenya et la Tanzanie. Les trois absences de 2020 restent manquantes. L'unité `persons` représente ici les arrivées selon la définition de la source, pas nécessairement des visiteurs uniques.

### 4.1 Évolution globale

**Question :** comment évoluent les arrivées sur l'ensemble de la période observée, et quelles différences d'échelle apparaissent ? Le graphique juxtapose les séries nationales, sans les sommer. Chaque séquence d'années consécutives forme un segment distinct : aucun trou n'est relié artificiellement.

In [18]:
def segment_arrivals(table):
    """Prépare des segments contigus à partir des seules valeurs renseignées."""
    result = table.loc[table["value"].notna()].sort_values(["destination", "year"]).copy()
    result["segment"] = result.groupby("destination")["year"].transform(
        lambda years: years.diff().ne(1).cumsum()
    )
    return result


arrivals_plot = segment_arrivals(arrivals_available)
history_start = int(arrivals_available["year"].min())
history_end = int(arrivals_available["year"].max())
history_chart = alt.Chart(arrivals_plot).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Année", axis=alt.Axis(format="d", tickMinStep=1)),
    y=alt.Y("value:Q", title="Arrivées internationales (personnes)", axis=alt.Axis(format=",.0f")),
    color=alt.Color("destination:N", title="Destination", scale=arrival_colors, legend=alt.Legend(orient="bottom", columns=3)),
    detail=alt.Detail("segment:N"),
    order=alt.Order("year:Q"),
    tooltip=[
        alt.Tooltip("destination:N", title="Destination"),
        alt.Tooltip("year:Q", title="Année", format="d"),
        alt.Tooltip("value:Q", title="Arrivées (personnes)", format=",.0f"),
        alt.Tooltip("quality_flag:N", title="Qualité"),
    ],
).properties(
    width=760, height=360,
    title=alt.Title(f"Arrivées internationales — {history_start}–{history_end}", subtitle=[arrival_source_note, "Égypte, Kenya, Tanzanie : aucune valeur renseignée en 2020."]),
)
arrival_charts.append(history_chart)
display(history_chart)

alt.Chart(...)

**Interprétation.** Entre 1995 et 2019, les niveaux augmentent pour les sept destinations, avec des fluctuations intermédiaires. Les échelles diffèrent fortement : Afrique du Sud, Maroc et Égypte atteignent des volumes bien supérieurs à Maurice, au Kenya et à la Tanzanie. Une rupture descendante est visible en 2020 pour les quatre séries renseignées ; elle ne peut pas être mesurée cette année-là pour les trois autres. Ces trajectoires ne démontrent pas leurs causes.

### 4.2 Comparaison des 7 destinations

**Question :** quels volumes observe-t-on lors de la dernière année réellement commune ? L'année est calculée par intersection des années renseignées. Le rang porte uniquement sur les sept valeurs de cette année ; les ex æquo éventuels reçoivent le même rang.

In [19]:
arrival_year_sets = [
    set(arrivals_available.loc[arrivals_available["destination"].eq(d), "year"])
    for d in DESTINATIONS
]
arrival_common_years = set.intersection(*arrival_year_sets)
assert arrival_common_years, "Aucune année commune : classement impossible."
arrival_common_year = int(max(arrival_common_years))
arrival_comparison = arrivals_available.loc[
    arrivals_available["year"].eq(arrival_common_year), ["destination", "value"]
].rename(columns={"value": "arrivees"}).copy()
assert len(arrival_comparison) == len(DESTINATIONS)
assert arrival_comparison["destination"].nunique() == len(DESTINATIONS)
arrival_comparison["rang"] = arrival_comparison["arrivees"].rank(method="min", ascending=False).astype(int)
arrival_comparison = arrival_comparison.sort_values(["rang", "destination"]).reset_index(drop=True)
display(Markdown(f"**Année commune calculée : {arrival_common_year}.**"))
show_arrivals_table(arrival_comparison, counts=["arrivees"])
comparison_chart = alt.Chart(arrival_comparison).mark_bar().encode(
    y=alt.Y("destination:N", title="Destination", sort="-x"),
    x=alt.X("arrivees:Q", title="Arrivées internationales (personnes)", axis=alt.Axis(format=",.0f")),
    color=alt.Color("destination:N", scale=arrival_colors, legend=None),
    tooltip=[alt.Tooltip("destination:N", title="Destination"), alt.Tooltip("arrivees:Q", title="Arrivées", format=",.0f"), alt.Tooltip("rang:Q", title="Rang", format="d")],
).properties(width=700, height=280, title=alt.Title(
    f"Sept destinations à période comparable — {arrival_common_year}", subtitle=arrival_source_note,
))
arrival_charts.append(comparison_chart)
display(comparison_chart)

**Année commune calculée : 2019.**

,destination,arrivees,rang
0,Afrique du Sud,14 797 000,1
1,Maroc,13 109 000,2
2,Égypte,13 026 000,3
3,Tunisie,9 429 000,4
4,Kenya,2 049 000,5
5,Tanzanie,1 527 000,6
6,Maurice,1 418 000,7


alt.Chart(...)

**Interprétation.** En 2019, l'Afrique du Sud arrive en tête avec **14 797 000** arrivées, suivie du Maroc (**13 109 000**) et de l'Égypte (**13 026 000**). Maurice (**1 418 000**) et la Tanzanie (**1 527 000**) ont les niveaux les plus faibles parmi les sept destinations. Ce classement décrit un volume sur une année commune ; il ne résume ni toute la trajectoire historique ni la performance touristique au sens large.

### 4.3 Tendances avant 2020

**Question :** quelle croissance moyenne observe-t-on sur une période commune avant 2020 ?

Retenir les première et dernière années communes renseignées strictement antérieures à 2020. Le CAGR est défini par `(valeur_fin / valeur_debut) ** (1 / (année_fin - année_debut)) - 1`. Le nombre d'années désigne les intervalles écoulés, pas le nombre d'observations. Les bornes doivent être renseignées et strictement positives. Le CAGR résume les deux bornes ; il ne suppose pas une croissance régulière et ne sert pas à interpoler les années.

In [20]:
pre_common_years = sorted(y for y in arrival_common_years if y < 2020)
assert len(pre_common_years) >= 2, "Période commune insuffisante pour un CAGR."
pre_start, pre_end = int(pre_common_years[0]), int(pre_common_years[-1])
pre_duration = pre_end - pre_start
assert pre_duration > 0
arrival_wide = arrivals_available.pivot(index="destination", columns="year", values="value").reindex(DESTINATIONS)
pre_trends = arrival_wide[[pre_start, pre_end]].rename(columns={
    pre_start: "arrivees_debut", pre_end: "arrivees_fin",
}).copy()
pre_trends["annee_debut"] = pre_start
pre_trends["annee_fin"] = pre_end
pre_trends["nombre_annees"] = pre_duration
valid_endpoints = pre_trends[["arrivees_debut", "arrivees_fin"]].notna().all(axis=1) & pre_trends[["arrivees_debut", "arrivees_fin"]].gt(0).all(axis=1)
pre_trends["cagr_pct"] = np.nan
pre_trends.loc[valid_endpoints, "cagr_pct"] = (
    (pre_trends.loc[valid_endpoints, "arrivees_fin"] / pre_trends.loc[valid_endpoints, "arrivees_debut"]) ** (1 / pre_duration) - 1
) * 100
pre_trends["statut"] = valid_endpoints.map({True: "Calculable", False: "Non calculable"})
pre_trends = pre_trends.reset_index()
display(Markdown(f"**Période commune : {pre_start}–{pre_end}, soit {pre_duration} années écoulées.**"))
show_arrivals_table(pre_trends, counts=["arrivees_debut", "arrivees_fin"], percentages=["cagr_pct"])
cagr_chart = alt.Chart(pre_trends.loc[valid_endpoints.to_numpy()]).mark_bar().encode(
    y=alt.Y("destination:N", sort="-x", title="Destination"),
    x=alt.X("cagr_pct:Q", title="CAGR des arrivées (% par an)", axis=alt.Axis(format=".2f")),
    tooltip=[alt.Tooltip("destination:N", title="Destination"), alt.Tooltip("cagr_pct:Q", title="CAGR (%/an)", format=".2f")],
).properties(width=700, height=280, title=alt.Title(
    f"Croissance annualisée entre les bornes communes — {pre_start}–{pre_end}",
    subtitle="Source : WDI ; synthèse des bornes, sans hypothèse de croissance régulière.",
))
arrival_charts.append(cagr_chart)
display(cagr_chart)

**Période commune : 1995–2019, soit 24 années écoulées.**

year,destination,arrivees_debut,arrivees_fin,annee_debut,annee_fin,nombre_annees,cagr_pct,statut
0,Afrique du Sud,4 684 000,14 797 000,1995,2019,24,"+4,91 %",Calculable
1,Égypte,3 133 000,13 026 000,1995,2019,24,"+6,12 %",Calculable
2,Kenya,974 000,2 049 000,1995,2019,24,"+3,15 %",Calculable
3,Maroc,2 752 000,13 109 000,1995,2019,24,"+6,72 %",Calculable
4,Maurice,437 000,1 418 000,1995,2019,24,"+5,03 %",Calculable
5,Tanzanie,295 000,1 527 000,1995,2019,24,"+7,09 %",Calculable
6,Tunisie,4 120 000,9 429 000,1995,2019,24,"+3,51 %",Calculable


alt.Chart(...)

**Interprétation.** Sur **1995–2019, soit 24 ans**, les sept CAGR sont positifs. La Tanzanie (**7,09 % par an**) et le Maroc (**6,72 %**) présentent les hausses annualisées les plus élevées, le Kenya (**3,15 %**) la plus faible. La croissance relative ne mesure pas le volume absolu : elle dépend aussi du niveau initial et masque les fluctuations entre les bornes.

### 4.4 Rupture de 2020

**Question :** quelle rupture observe-t-on entre 2019 et 2020 lorsque les deux valeurs existent et sont strictement positives ?

Calculs : `variation_absolue = arrivals_2020 - arrivals_2019` ; `variation_pct = (arrivals_2020 / arrivals_2019 - 1) × 100`. Les destinations non calculables restent dans le tableau avec des résultats manquants et un statut explicite. Le graphique n'affiche que les variations calculables.

In [21]:
arrival_break = arrival_wide.reindex(columns=[2019, 2020]).rename(columns={
    2019: "arrivals_2019", 2020: "arrivals_2020",
}).copy()
break_columns = ["arrivals_2019", "arrivals_2020"]
break_valid = arrival_break[break_columns].notna().all(axis=1) & arrival_break[break_columns].gt(0).all(axis=1)
arrival_break["variation_absolue"] = np.nan
arrival_break["variation_pct"] = np.nan
arrival_break.loc[break_valid, "variation_absolue"] = arrival_break.loc[break_valid, "arrivals_2020"] - arrival_break.loc[break_valid, "arrivals_2019"]
arrival_break.loc[break_valid, "variation_pct"] = (arrival_break.loc[break_valid, "arrivals_2020"] / arrival_break.loc[break_valid, "arrivals_2019"] - 1) * 100
arrival_break["statut"] = break_valid.map({True: "Calculable", False: "Non calculable"})
arrival_break["raison"] = [
    "Deux valeurs renseignées et positives" if ok else
    "Valeur 2019 ou 2020 manquante" if row[break_columns].isna().any() else
    "Valeur nulle ou négative : calcul exclu"
    for (_, row), ok in zip(arrival_break.iterrows(), break_valid)
]
arrival_break = arrival_break.reset_index()
show_arrivals_table(arrival_break, counts=break_columns + ["variation_absolue"], percentages=["variation_pct"])
break_plot = arrival_break.loc[arrival_break["statut"].eq("Calculable")]
assert not break_plot.empty, "Aucune variation calculable : ne pas produire de graphique vide."
break_chart = alt.Chart(break_plot).mark_bar(color="#b54a45").encode(
    y=alt.Y("destination:N", title="Destination", sort=DESTINATIONS),
    x=alt.X("variation_pct:Q", title="Variation des arrivées 2019–2020 (%)", axis=alt.Axis(format=".1f")),
    tooltip=[alt.Tooltip("destination:N", title="Destination"), alt.Tooltip("variation_pct:Q", title="Variation (%)", format=".2f"), alt.Tooltip("variation_absolue:Q", title="Variation (personnes)", format=",.0f")],
).properties(width=700, height=250, title=alt.Title(
    "Rupture observée entre 2019 et 2020",
    subtitle="Source : WDI ; seules les destinations avec deux valeurs positives sont représentées.",
))
arrival_charts.append(break_chart)
display(break_chart)

year,destination,arrivals_2019,arrivals_2020,variation_absolue,variation_pct,statut,raison
0,Afrique du Sud,14 797 000,3 886 600,-10 910 400,"-73,73 %",Calculable,Deux valeurs renseignées et positives
1,Égypte,13 026 000,Non renseigné,Non renseigné,Non renseigné,Non calculable,Valeur 2019 ou 2020 manquante
2,Kenya,2 049 000,Non renseigné,Non renseigné,Non renseigné,Non calculable,Valeur 2019 ou 2020 manquante
3,Maroc,13 109 000,2 802 000,-10 307 000,"-78,63 %",Calculable,Deux valeurs renseignées et positives
4,Maurice,1 418 000,316 000,-1 102 000,"-77,72 %",Calculable,Deux valeurs renseignées et positives
5,Tanzanie,1 527 000,Non renseigné,Non renseigné,Non renseigné,Non calculable,Valeur 2019 ou 2020 manquante
6,Tunisie,9 429 000,2 012 000,-7 417 000,"-78,66 %",Calculable,Deux valeurs renseignées et positives


alt.Chart(...)

**Interprétation.** La rupture observée entre 2019 et 2020 est de **−73,73 % en Afrique du Sud**, **−78,63 % au Maroc**, **−77,72 % à Maurice** et **−78,66 % en Tunisie**. Égypte, Kenya et Tanzanie sont non calculables faute de valeur 2020. L'année 2020 s'inscrit dans le contexte Covid ; le dataset seul ne permet pas d'attribuer causalement toute la variation à ce contexte.

### 4.5 Limite sur la reprise post-Covid

**Question :** dispose-t-on d'arrivées nationales renseignées après 2020 ? Afficher la dernière année renseignée de chaque destination et compter les valeurs postérieures à 2020. Ce contrôle de couverture ne calcule aucun indicateur de reprise.

In [22]:
arrival_latest = arrivals_available.loc[
    arrivals_available.groupby("destination")["year"].idxmax(),
    ["destination", "year", "value", "quality_flag"],
].set_index("destination").reindex(DESTINATIONS).rename(columns={
    "year": "derniere_annee_disponible", "value": "arrivees",
})
arrival_post_coverage = arrival_latest[["derniere_annee_disponible"]].copy()
# Compte de disponibilité ; un zéro ici signifie zéro observation, pas zéro arrivée.
arrival_post_coverage["valeurs_renseignees_apres_2020"] = [
    int((arrivals_available["destination"].eq(d) & arrivals_available["year"].gt(2020)).sum())
    for d in DESTINATIONS
]
display(arrival_post_coverage)
arrival_post_count = int(arrival_post_coverage["valeurs_renseignees_apres_2020"].sum())
display(Markdown(f"**Couverture recalculée : {arrival_post_count} valeur nationale d'arrivées renseignée après 2020.**"))
assert arrival_post_count == 0, "La couverture a évolué : réviser la limite documentaire avant d'analyser la reprise."

,derniere_annee_disponible,valeurs_renseignees_apres_2020
destination,,
Afrique du Sud,2020,0
Égypte,2019,0
Kenya,2019,0
Maroc,2020,0
Maurice,2020,0
Tanzanie,2019,0
Tunisie,2020,0


**Couverture recalculée : 0 valeur nationale d'arrivées renseignée après 2020.**

**Interprétation.** Aucune série nationale d'arrivées ne possède de valeur renseignée après 2020 : une analyse de reprise ou de retour au niveau pré-Covid n'est pas réalisable. Les provenances 2022–2024 ont d'autres périmètres — panels, Top N ou parts — et ne remplacent pas ces totaux. Un enrichissement documenté des séries nationales serait nécessaire ; aucune reconstruction n'est effectuée.

### 4.6 Niveaux les plus récents disponibles

**Question :** quelle est la dernière valeur connue de chaque destination ? Présenter les destinations dans l'ordre du projet, sans classement par volume. Cette vue diffère du classement à année commune de 4.2.

**Ces niveaux ne sont pas strictement comparables lorsque les années diffèrent.** « Récent » signifie ici dernière observation du dataset, et non situation actuelle.

In [23]:
show_arrivals_table(arrival_latest.reset_index(), counts=["arrivees"])
display(Markdown(
    f"Années propres aux destinations : **{', '.join(map(str, sorted(arrival_latest['derniere_annee_disponible'].unique())))}**. "
    f"Pour une comparaison à année identique, utiliser la vue **{arrival_common_year}** de la section 4.2."
))

,destination,derniere_annee_disponible,arrivees,quality_flag
0,Afrique du Sud,2020,3 886 600,available
1,Égypte,2019,13 026 000,available
2,Kenya,2019,2 049 000,available
3,Maroc,2020,2 802 000,available
4,Maurice,2020,316 000,available
5,Tanzanie,2019,1 527 000,available
6,Tunisie,2020,2 012 000,available


Années propres aux destinations : **2019, 2020**. Pour une comparaison à année identique, utiliser la vue **2019** de la section 4.2.

**Interprétation.** Les derniers niveaux sont ceux de 2020 pour l'Afrique du Sud, le Maroc, Maurice et la Tunisie, et ceux de 2019 pour l'Égypte, le Kenya et la Tanzanie. Comparer directement ces niveaux mélangerait une année précédant la rupture et l'année de rupture. Le tableau conserve les années et les drapeaux de qualité ; aucun rang n'est attribué.

In [24]:
# Contrôles finaux de la section 04 : aucune modification des données sources.
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
pd.testing.assert_frame_equal(arrivals_df, arrivals_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
assert arrivals_df["value"].isna().sum() == 3
assert len(arrivals_available) == 179 and len(arrivals_df) == 182
assert arrival_common_year == max(set.intersection(*arrival_year_sets))
assert set(arrival_comparison["destination"]) == set(DESTINATIONS)
assert arrival_break.loc[arrival_break["statut"].eq("Non calculable"), ["variation_absolue", "variation_pct"]].isna().all().all()
assert set(break_plot["destination"]) == set(arrivals_available.loc[arrivals_available["year"].eq(2019), "destination"]) & set(arrivals_available.loc[arrivals_available["year"].eq(2020), "destination"])
assert arrival_latest["derniere_annee_disponible"].equals(arrivals_available.groupby("destination")["year"].max().reindex(DESTINATIONS).rename("derniere_annee_disponible"))
assert "rang" not in arrival_latest.columns
assert len(arrival_charts) == 4
for chart in arrival_charts:
    spec = chart.to_dict(validate=True)
    assert spec.get("datasets") and all(len(rows) > 0 for rows in spec["datasets"].values())
# Vérifie aussi le traitement d'un trou interne sur une copie privée de contrôle.
probe = arrivals_available.loc[arrivals_available["destination"].eq(DESTINATIONS[0])]
middle_year = int(probe["year"].sort_values().iloc[2])
segmented_probe = segment_arrivals(probe.loc[probe["year"].ne(middle_year)])
assert segmented_probe.loc[segmented_probe["year"].eq(middle_year - 1), "segment"].iloc[0] != segmented_probe.loc[segmented_probe["year"].eq(middle_year + 1), "segment"].iloc[0]
print("Section 04 validée : données intactes, périodes explicites, absences conservées, quatre graphiques non vides.")

Section 04 validée : données intactes, périodes explicites, absences conservées, quatre graphiques non vides.


**Fin de la phase 2.** L'analyse descriptive des arrivées est limitée aux données nationales renseignées. Les recettes, les analyses croisées, les marchés d'origine et les KPI finaux du dashboard restent hors de cette phase.

## 05 — Analyse des recettes touristiques

**Objectif :** décrire les recettes nationales à partir de la seule couche `receipts` déjà chargée dans `df`. Les calculs utilisent des valeurs renseignées et des périodes explicites, sans reconstruction depuis les arrivées ou les provenances.

> **Unité : USD courants (`current_USD`).** Les niveaux et croissances sont nominaux. Une hausse ne mesure pas directement une hausse du pouvoir d'achat ou de la performance économique réelle. Aucune correction d'inflation, interpolation ou extrapolation n'est effectuée.

**Préparation :** contrôler les sept destinations, les unités, les métadonnées et les couvertures. Les fonctions de formatage et de segmentation de la section 04 sont réutilisées sans modifier ses cellules.

In [25]:
receipts_df = df.loc[df["dataset_layer"].eq("receipts")].copy()
receipts_initial = receipts_df.copy(deep=True)
receipts_available = receipts_df.loc[receipts_df["value"].notna()].copy()
assert receipts_df["dataset_layer"].eq("receipts").all()
assert set(receipts_df["destination"]) == set(DESTINATIONS)
assert not receipts_df.duplicated(["destination", "year"]).any()
for column, expected in {
    "unit": "current_USD", "metric": "tourism_receipts",
    "metric_type": "destination_total", "granularity": "destination_total",
}.items():
    assert receipts_df[column].eq(expected).all(), f"Recettes : {column} incohérent."
assert set(receipts_df["quality_flag"]) <= {"available", "missing_in_source"}
assert receipts_df["value"].isna().eq(receipts_df["quality_flag"].eq("missing_in_source")).all()
assert np.isfinite(receipts_available["value"]).all()
assert receipts_available["value"].ge(0).all(), "Valeur négative : vérifier la source."
receipts_coverage = receipts_df.groupby("destination").agg(
    premiere_ligne=("year", "min"), derniere_ligne=("year", "max"),
    lignes=("value", "size"), valeurs_renseignees=("value", "count"),
    valeurs_manquantes=("value", lambda s: s.isna().sum()),
).join(receipts_available.groupby("destination").agg(
    premiere_annee_renseignee=("year", "min"), derniere_annee_renseignee=("year", "max"),
)).reindex(DESTINATIONS)
display(receipts_coverage)
display(receipts_df[["unit", "metric", "metric_type", "quality_flag", "source_name", "source_reference", "coverage_scope"]].drop_duplicates())
display(receipts_df.loc[receipts_df["value"].isna(), ["destination", "year", "value", "quality_flag"]])

# Ces fonctions ne dépendent pas de l'indicateur malgré leurs noms historiques.
segment_receipts = segment_arrivals
receipts_colors = arrival_colors
receipts_source_note = "Source : World Bank WDI — ST.INT.RCPT.CD ; USD courants."
receipts_charts = []


def show_receipts_table(table, amounts=(), billions=(), percentages=()):
    formats = {c: format_count for c in amounts}
    formats.update({c: (lambda v: f"{v:,.3f}".replace(",", " ").replace(".", ",") if pd.notna(v) else "Non renseigné") for c in billions})
    formats.update({c: format_percent for c in percentages})
    display(table.style.format(formats, na_rep="Non renseigné"))


def receipts_bar(table, field, axis_title, title, subtitle, percentage=False):
    """Même convention de barres pour niveau commun, CAGR nominal et rupture."""
    assert not table.empty, "Ne pas produire de graphique vide."
    chart = alt.Chart(table).mark_bar().encode(
        y=alt.Y("destination:N", title="Destination", sort="-x"),
        x=alt.X(f"{field}:Q", title=axis_title, axis=alt.Axis(format=".2f")),
        color=alt.Color("destination:N", scale=receipts_colors, legend=None),
        tooltip=[
            alt.Tooltip("destination:N", title="Destination"),
            alt.Tooltip(f"{field}:Q", title=axis_title, format=".2f" if percentage else ".3f"),
        ],
    ).properties(width=700, height=280, title=alt.Title(title, subtitle=subtitle))
    receipts_charts.append(chart)
    display(chart)

,premiere_ligne,derniere_ligne,lignes,valeurs_renseignees,valeurs_manquantes,premiere_annee_renseignee,derniere_annee_renseignee
destination,,,,,,,
Afrique du Sud,1995,2020,26,26,0,1995,2020
Égypte,1995,2020,26,26,0,1995,2020
Kenya,1995,2020,26,25,1,1995,2019
Maroc,1995,2020,26,26,0,1995,2020
Maurice,1995,2020,26,26,0,1995,2020
Tanzanie,1995,2020,26,23,3,1997,2019
Tunisie,1995,2020,26,26,0,1995,2020


,unit,metric,metric_type,quality_flag,source_name,source_reference,coverage_scope
182,current_USD,tourism_receipts,destination_total,available,World Bank WDI,WB_WDI_ST_INT_RCPT_CD,Common WDI destination-level series
259,current_USD,tourism_receipts,destination_total,missing_in_source,World Bank WDI,WB_WDI_ST_INT_RCPT_CD,Common WDI destination-level series


,destination,year,value,quality_flag
259,Kenya,2020,NaN,missing_in_source
312,Tanzanie,1995,NaN,missing_in_source
313,Tanzanie,1996,NaN,missing_in_source
337,Tanzanie,2020,NaN,missing_in_source


**Interprétation.** Les recettes comportent 182 lignes, dont 178 valeurs renseignées. Les quatre absences sont celles de la Tanzanie en 1995, 1996 et 2020, et du Kenya en 2020. Les autres destinations disposent de valeurs sur 1995–2020. Leurs montants restent en USD courants dans les données.

### 5.1 Évolution globale

**Question :** comment évoluent les niveaux nominaux des recettes sur leur période réelle ? Afficher les montants en milliards USD courants dans une copie destinée au graphique (`display_value = value / 1_000_000_000`). Les valeurs originales restent intactes ; chaque séquence d'années consécutives forme un segment distinct.

In [26]:
receipts_plot = segment_receipts(receipts_available)
receipts_plot["display_value"] = receipts_plot["value"] / 1_000_000_000
receipts_history_start = int(receipts_available["year"].min())
receipts_history_end = int(receipts_available["year"].max())
receipts_history_chart = alt.Chart(receipts_plot).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Année", axis=alt.Axis(format="d", tickMinStep=1)),
    y=alt.Y("display_value:Q", title="Recettes touristiques — milliards USD courants", axis=alt.Axis(format=".1f")),
    color=alt.Color("destination:N", title="Destination", scale=receipts_colors, legend=alt.Legend(orient="bottom", columns=3)),
    detail=alt.Detail("segment:N"), order=alt.Order("year:Q"),
    tooltip=[
        alt.Tooltip("destination:N", title="Destination"),
        alt.Tooltip("year:Q", title="Année", format="d"),
        alt.Tooltip("display_value:Q", title="Milliards USD courants", format=".3f"),
        alt.Tooltip("value:Q", title="USD courants (source)", format=",.0f"),
        alt.Tooltip("quality_flag:N", title="Qualité"),
    ],
).properties(width=760, height=360, title=alt.Title(
    f"Recettes touristiques nominales — {receipts_history_start}–{receipts_history_end}",
    subtitle=[receipts_source_note, "Tanzanie : début renseigné en 1997 ; Kenya et Tanzanie : 2020 manquant."],
))
receipts_charts.append(receipts_history_chart)
display(receipts_history_chart)

alt.Chart(...)

**Interprétation.** Les niveaux nominaux de 2019 dépassent ceux de 1997 dans les sept destinations, avec des fluctuations intermédiaires. En 2019, l'Égypte, le Maroc et l'Afrique du Sud présentent les niveaux les plus élevés. Une baisse apparaît en 2020 dans les cinq séries renseignées. Ces évolutions en USD courants ne mesurent pas une croissance réelle et ne démontrent pas leurs causes.

### 5.2 Comparaison des 7 destinations

**Question :** quelle est la dernière année où les sept destinations possèdent une recette renseignée ? Calculer cette année exclusivement depuis les recettes. Le classement porte sur cette année commune ; les colonnes en milliards servent à la lisibilité.

In [27]:
receipts_year_sets = [
    set(receipts_available.loc[receipts_available["destination"].eq(d), "year"])
    for d in DESTINATIONS
]
receipts_common_years = set.intersection(*receipts_year_sets)
assert receipts_common_years, "Aucune année commune aux recettes."
receipts_common_year = int(max(receipts_common_years))
receipts_comparison = receipts_available.loc[
    receipts_available["year"].eq(receipts_common_year), ["destination", "value"]
].rename(columns={"value": "receipts"}).copy()
assert len(receipts_comparison) == len(DESTINATIONS)
assert receipts_comparison["destination"].nunique() == len(DESTINATIONS)
receipts_comparison["receipts_billions"] = receipts_comparison["receipts"] / 1_000_000_000
receipts_comparison["rang"] = receipts_comparison["receipts"].rank(method="min", ascending=False).astype(int)
receipts_comparison = receipts_comparison.sort_values(["rang", "destination"]).reset_index(drop=True)
display(Markdown(f"**Dernière année commune calculée pour les recettes : {receipts_common_year}. Montants en USD courants.**"))
show_receipts_table(receipts_comparison, amounts=["receipts"], billions=["receipts_billions"])
receipts_bar(receipts_comparison, "receipts_billions", "Recettes — milliards USD courants",
             f"Recettes comparées sur une année commune — {receipts_common_year}", receipts_source_note)

**Dernière année commune calculée pour les recettes : 2019. Montants en USD courants.**

,destination,receipts,receipts_billions,rang
0,Égypte,14 256 000 000,"14,256",1
1,Maroc,9 949 000 000,"9,949",2
2,Afrique du Sud,9 064 000 000,"9,064",3
3,Tunisie,2 683 000 000,"2,683",4
4,Tanzanie,2 624 500 000,"2,624",5
5,Maurice,2 024 000 000,"2,024",6
6,Kenya,1 762 000 000,"1,762",7


alt.Chart(...)

**Interprétation.** En **2019**, l'Égypte atteint **14,256 milliards USD courants**, devant le Maroc (**9,949**) et l'Afrique du Sud (**9,064**). Les niveaux les plus faibles sont ceux du Kenya (**1,762**) et de Maurice (**2,024**). Ce classement de recettes nominales ne constitue pas un jugement global de performance économique.

### 5.3 Tendances avant 2020

**Question :** quelle croissance nominale annualisée observe-t-on entre deux bornes communes avant 2020 ?

Les première et dernière années communes renseignées avant 2020 sont calculées depuis les recettes. Formule : `CAGR = (recettes_fin / recettes_debut) ** (1 / (annee_fin - annee_debut)) - 1`. Les bornes doivent être renseignées et strictement positives. Le tableau affiche le CAGR en pourcentage par an. Il résume les bornes, sans interpolation, et n'est **pas corrigé de l'inflation**.

In [28]:
receipts_pre_years = sorted(y for y in receipts_common_years if y < 2020)
assert len(receipts_pre_years) >= 2, "Période commune insuffisante pour un CAGR."
receipts_pre_start, receipts_pre_end = int(receipts_pre_years[0]), int(receipts_pre_years[-1])
receipts_duration = receipts_pre_end - receipts_pre_start
assert receipts_duration > 0
receipts_wide = receipts_available.pivot(index="destination", columns="year", values="value").reindex(DESTINATIONS)
receipts_trends = receipts_wide[[receipts_pre_start, receipts_pre_end]].rename(columns={
    receipts_pre_start: "recettes_debut", receipts_pre_end: "recettes_fin",
}).copy()
receipts_trends["annee_debut"] = receipts_pre_start
receipts_trends["annee_fin"] = receipts_pre_end
receipts_trends["nombre_annees"] = receipts_duration
receipts_endpoints_valid = receipts_trends[["recettes_debut", "recettes_fin"]].notna().all(axis=1) & receipts_trends[["recettes_debut", "recettes_fin"]].gt(0).all(axis=1)
receipts_trends["CAGR"] = np.nan
receipts_trends.loc[receipts_endpoints_valid, "CAGR"] = (
    (receipts_trends.loc[receipts_endpoints_valid, "recettes_fin"] / receipts_trends.loc[receipts_endpoints_valid, "recettes_debut"]) ** (1 / receipts_duration) - 1
) * 100
receipts_trends["statut"] = receipts_endpoints_valid.map({True: "Calculable", False: "Non calculable"})
receipts_trends = receipts_trends.reset_index()[[
    "destination", "annee_debut", "recettes_debut", "annee_fin", "recettes_fin", "nombre_annees", "CAGR", "statut",
]]
display(Markdown(f"**Période commune : {receipts_pre_start}–{receipts_pre_end}, soit {receipts_duration} ans. CAGR nominal en %/an ; bornes en USD courants.**"))
show_receipts_table(receipts_trends, amounts=["recettes_debut", "recettes_fin"], percentages=["CAGR"])
receipts_bar(receipts_trends.loc[receipts_trends["statut"].eq("Calculable")], "CAGR",
             "CAGR nominal des recettes (% par an)",
             f"Croissance nominale entre bornes communes — {receipts_pre_start}–{receipts_pre_end}",
             "Source : WDI ; USD courants, sans correction d'inflation.", percentage=True)

**Période commune : 1997–2019, soit 22 ans. CAGR nominal en %/an ; bornes en USD courants.**

year,destination,annee_debut,recettes_debut,annee_fin,recettes_fin,nombre_annees,CAGR,statut
0,Afrique du Sud,1997,3 422 000 000,2019,9 064 000 000,22,"+4,53 %",Calculable
1,Égypte,1997,4 046 000 000,2019,14 256 000 000,22,"+5,89 %",Calculable
2,Kenya,1997,1 077 000 000,2019,1 762 000 000,22,"+2,26 %",Calculable
3,Maroc,1997,1 649 000 000,2019,9 949 000 000,22,"+8,51 %",Calculable
4,Maurice,1997,666 000 000,2019,2 024 000 000,22,"+5,18 %",Calculable
5,Tanzanie,1997,343 000 000,2019,2 624 500 000,22,"+9,69 %",Calculable
6,Tunisie,1997,1 858 000 000,2019,2 683 000 000,22,"+1,68 %",Calculable


alt.Chart(...)

**Interprétation.** La période commune est **1997–2019, soit 22 ans**, car les recettes tanzaniennes manquent en 1995–1996. Tous les CAGR nominaux sont positifs ; les plus élevés sont ceux de la Tanzanie (**9,69 %/an**) et du Maroc (**8,51 %/an**), le plus faible celui de la Tunisie (**1,68 %/an**). Ces taux en USD courants ne mesurent pas la croissance réelle et masquent les fluctuations entre les bornes.

### 5.4 Rupture de 2020

**Question :** quelle rupture observe-t-on entre 2019 et 2020 quand les deux montants sont renseignés ?

`variation_absolue = receipts_2020 - receipts_2019` en USD courants ; `variation_pct = (receipts_2020 / receipts_2019 - 1) × 100`. Le dénominateur 2019 doit être strictement positif. Un éventuel zéro observé en 2020 resterait une valeur valide ; il ne serait jamais créé à partir d'une absence. Les couples incomplets restent « Non calculable ».

In [29]:
receipts_break = receipts_wide.reindex(columns=[2019, 2020]).rename(columns={
    2019: "receipts_2019", 2020: "receipts_2020",
}).copy()
receipts_break_cols = ["receipts_2019", "receipts_2020"]
receipts_break_valid = receipts_break[receipts_break_cols].notna().all(axis=1) & receipts_break["receipts_2019"].gt(0)
receipts_break["variation_absolue"] = np.nan
receipts_break["variation_pct"] = np.nan
receipts_break.loc[receipts_break_valid, "variation_absolue"] = receipts_break.loc[receipts_break_valid, "receipts_2020"] - receipts_break.loc[receipts_break_valid, "receipts_2019"]
receipts_break.loc[receipts_break_valid, "variation_pct"] = (receipts_break.loc[receipts_break_valid, "receipts_2020"] / receipts_break.loc[receipts_break_valid, "receipts_2019"] - 1) * 100
receipts_break["statut"] = receipts_break_valid.map({True: "Calculable", False: "Non calculable"})
receipts_break["raison"] = [
    "Couple renseigné, référence 2019 positive" if valid else
    "Valeur 2019 ou 2020 manquante" if row[receipts_break_cols].isna().any() else
    "Référence 2019 non positive"
    for (_, row), valid in zip(receipts_break.iterrows(), receipts_break_valid)
]
receipts_break = receipts_break.reset_index()
show_receipts_table(receipts_break, amounts=receipts_break_cols + ["variation_absolue"], percentages=["variation_pct"])
receipts_break_plot = receipts_break.loc[receipts_break["statut"].eq("Calculable")]
receipts_bar(receipts_break_plot, "variation_pct", "Variation nominale des recettes 2019–2020 (%)",
             "Rupture observée entre 2019 et 2020",
             "Source : WDI ; USD courants, couples renseignés uniquement.", percentage=True)

year,destination,receipts_2019,receipts_2020,variation_absolue,variation_pct,statut,raison
0,Afrique du Sud,9 064 000 000,2 716 000 000,-6 348 000 000,"-70,04 %",Calculable,"Couple renseigné, référence 2019 positive"
1,Égypte,14 256 000 000,4 874 000 000,-9 382 000 000,"-65,81 %",Calculable,"Couple renseigné, référence 2019 positive"
2,Kenya,1 762 000 000,Non renseigné,Non renseigné,Non renseigné,Non calculable,Valeur 2019 ou 2020 manquante
3,Maroc,9 949 000 000,4 514 000 000,-5 435 000 000,"-54,63 %",Calculable,"Couple renseigné, référence 2019 positive"
4,Maurice,2 024 000 000,518 000 000,-1 506 000 000,"-74,41 %",Calculable,"Couple renseigné, référence 2019 positive"
5,Tanzanie,2 624 500 000,Non renseigné,Non renseigné,Non renseigné,Non calculable,Valeur 2019 ou 2020 manquante
6,Tunisie,2 683 000 000,1 007 000 000,-1 676 000 000,"-62,47 %",Calculable,"Couple renseigné, référence 2019 positive"


alt.Chart(...)

**Interprétation.** La rupture observée entre 2019 et 2020 est de **−70,04 % en Afrique du Sud**, **−65,81 % en Égypte**, **−54,63 % au Maroc**, **−74,41 % à Maurice** et **−62,47 % en Tunisie**. Le Kenya et la Tanzanie sont non calculables faute de valeur 2020. Le contexte Covid est un repère temporel ; le dataset ne démontre pas une attribution causale de l'ensemble de ces variations.

### 5.5 Limite sur la reprise post-Covid

**Question :** existe-t-il des recettes nationales renseignées après 2020 ? Recalculer les dernières années et compter les valeurs post-2020, sans produire d'indicateur de reprise.

In [30]:
receipts_latest = receipts_available.loc[
    receipts_available.groupby("destination")["year"].idxmax(),
    ["destination", "year", "value", "quality_flag"],
].set_index("destination").reindex(DESTINATIONS).rename(columns={
    "year": "derniere_annee_disponible", "value": "recettes",
})
receipts_post_coverage = receipts_latest[["derniere_annee_disponible"]].rename(columns={
    "derniere_annee_disponible": "derniere_annee_recettes_renseignee",
})
receipts_post_coverage["valeurs_renseignees_apres_2020"] = [
    int((receipts_available["destination"].eq(d) & receipts_available["year"].gt(2020)).sum())
    for d in DESTINATIONS
]
display(receipts_post_coverage)
receipts_post_count = int(receipts_post_coverage["valeurs_renseignees_apres_2020"].sum())
display(Markdown(f"**Couverture recalculée : {receipts_post_count} valeur nationale de recettes renseignée après 2020.**"))
assert receipts_post_count == 0, "Couverture enrichie : réviser l'interprétation avant analyse de reprise."

,derniere_annee_recettes_renseignee,valeurs_renseignees_apres_2020
destination,,
Afrique du Sud,2020,0
Égypte,2020,0
Kenya,2019,0
Maroc,2020,0
Maurice,2020,0
Tanzanie,2019,0
Tunisie,2020,0


**Couverture recalculée : 0 valeur nationale de recettes renseignée après 2020.**

**Interprétation.** Il n'existe aucune valeur nationale de recettes après 2020. Étudier la vitesse de reprise, le retour au niveau 2019 ou l'évolution 2021–2024 nécessiterait un enrichissement documenté des séries nationales. Les provenances ne sont pas un substitut aux recettes ; aucune reconstruction ni extrapolation n'est réalisée.

### 5.6 Niveaux les plus récents disponibles

**Question :** comment distinguer la dernière valeur propre à chaque destination d'un niveau comparable sur une année commune ?

**A** conserve les années propres et ne classe pas les destinations. **B** réutilise exclusivement l'année commune de 5.2. Les montants restent en USD courants ; les colonnes en milliards sont des aides de lecture.

In [31]:
receipts_latest_view = receipts_latest.copy()
receipts_latest_view["recettes_milliards_USD"] = receipts_latest_view["recettes"] / 1_000_000_000
receipts_latest_view = receipts_latest_view.reset_index()[[
    "destination", "derniere_annee_disponible", "recettes", "recettes_milliards_USD", "quality_flag",
]]
display(Markdown("**A — Dernière valeur propre à chaque destination ; aucun rang.**"))
show_receipts_table(receipts_latest_view, amounts=["recettes"], billions=["recettes_milliards_USD"])
display(Markdown(f"**B — Année commune {receipts_common_year} ; classement sur une période identique.**"))
show_receipts_table(receipts_comparison, amounts=["receipts"], billions=["receipts_billions"])

**A — Dernière valeur propre à chaque destination ; aucun rang.**

,destination,derniere_annee_disponible,recettes,recettes_milliards_USD,quality_flag
0,Afrique du Sud,2020,2 716 000 000,"2,716",available
1,Égypte,2020,4 874 000 000,"4,874",available
2,Kenya,2019,1 762 000 000,"1,762",available
3,Maroc,2020,4 514 000 000,"4,514",available
4,Maurice,2020,518 000 000,"0,518",available
5,Tanzanie,2019,2 624 500 000,"2,624",available
6,Tunisie,2020,1 007 000 000,"1,007",available


**B — Année commune 2019 ; classement sur une période identique.**

,destination,receipts,receipts_billions,rang
0,Égypte,14 256 000 000,"14,256",1
1,Maroc,9 949 000 000,"9,949",2
2,Afrique du Sud,9 064 000 000,"9,064",3
3,Tunisie,2 683 000 000,"2,683",4
4,Tanzanie,2 624 500 000,"2,624",5
5,Maurice,2 024 000 000,"2,024",6
6,Kenya,1 762 000 000,"1,762",7


**Interprétation.** Les dernières recettes datent de 2020 pour cinq destinations, mais de 2019 pour le Kenya et la Tanzanie. Ces niveaux ne sont pas strictement comparables lorsque les années diffèrent. La vue commune **2019** répond à une autre question : comparer les sept montants au même moment. « Plus récent » désigne ici la dernière valeur du dataset, pas la situation actuelle.

In [32]:
# Contrôles finaux : identité des sources et validité des périmètres.
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
pd.testing.assert_frame_equal(receipts_df, receipts_initial, check_exact=True)
pd.testing.assert_frame_equal(receipts_df, df.loc[df["dataset_layer"].eq("receipts")], check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
assert receipts_df["unit"].eq("current_USD").all()
assert len(receipts_df) == 182 and len(receipts_available) == 178
assert receipts_df["value"].isna().sum() == 4
assert receipts_common_year == max(set.intersection(*receipts_year_sets))
assert receipts_common_year == int(receipts_available.groupby("year")["destination"].nunique().loc[lambda s: s.eq(len(DESTINATIONS))].index.max())
assert set(receipts_comparison["destination"]) == set(DESTINATIONS)
assert receipts_pre_start in receipts_common_years and receipts_pre_end in receipts_common_years
assert receipts_pre_end < 2020 and receipts_duration == receipts_pre_end - receipts_pre_start
assert receipts_trends.loc[receipts_trends["statut"].eq("Non calculable"), "CAGR"].isna().all()
assert receipts_break.loc[receipts_break["statut"].eq("Non calculable"), ["variation_absolue", "variation_pct"]].isna().all().all()
expected_break_destinations = set(receipts_available.loc[receipts_available["year"].eq(2019) & receipts_available["value"].gt(0), "destination"]) & set(receipts_available.loc[receipts_available["year"].eq(2020), "destination"])
assert set(receipts_break_plot["destination"]) == expected_break_destinations
assert "rang" not in receipts_latest_view.columns
assert receipts_latest["derniere_annee_disponible"].equals(receipts_available.groupby("destination")["year"].max().reindex(DESTINATIONS).rename("derniere_annee_disponible"))
np.testing.assert_array_equal(receipts_plot["display_value"].to_numpy(), receipts_plot["value"].to_numpy() / 1_000_000_000)
assert receipts_post_count == 0
assert len(receipts_charts) == 4
for chart in receipts_charts:
    chart_spec = chart.to_dict(validate=True)
    assert chart_spec.get("datasets") and all(len(rows) > 0 for rows in chart_spec["datasets"].values())
# La segmentation ne relie que des années consécutives, sur la seule couche receipts.
assert receipts_plot.groupby(["destination", "segment"])["year"].diff().dropna().eq(1).all()
print("Section 05 validée : USD courants, données intactes, périodes communes calculées, quatre graphiques non vides.")

Section 05 validée : USD courants, données intactes, périodes communes calculées, quatre graphiques non vides.


**Fin de la section 05.** L'analyse porte uniquement sur les recettes nationales nominales. L'analyse croisée arrivées × recettes, les provenances et les KPI définitifs restent hors de cette phase.

## 06 — Analyse croisée arrivées × recettes

**Objectif :** mettre en relation les deux séries nationales pour une même destination et une même année. Réutiliser `arrivals_df` et `receipts_df` sans nouveau chargement. Les unités sont distinctes : personnes pour les arrivées, USD courants pour les recettes. Le ratio est descriptif et ne représente pas une mesure individuelle.

### 6.1 Construction du panel comparable

**Question :** combien de couples destination × année possèdent simultanément les deux mesures ? Une jointure externe validée « un à un » conserve le périmètre potentiel ; le panel analytique retient ensuite seulement les couples complets. Les exclusions sont affichées, sans imputation.

In [33]:
cross_keys = ["destination", "year"]
cross_context = ["unit", "metric", "metric_type", "granularity", "source_name", "source_reference", "coverage_scope", "quality_flag", "notes"]
cross_arrivals_initial = arrivals_df.copy(deep=True)
cross_receipts_initial = receipts_df.copy(deep=True)
for table, layer, unit, metric in [
    (arrivals_df, "arrivals", "persons", "tourist_arrivals"),
    (receipts_df, "receipts", "current_USD", "tourism_receipts"),
]:
    assert table["dataset_layer"].eq(layer).all()
    assert table[cross_keys].notna().all().all()
    assert not table.duplicated(cross_keys).any(), "Jointure impossible : clé non unique."
    assert table["unit"].eq(unit).all() and table["metric"].eq(metric).all()
    assert table[["metric_type", "granularity"]].eq("destination_total").all().all()

cross_potential = arrivals_df[cross_keys + ["value"] + cross_context].rename(
    columns={"value": "arrivals"}
).merge(
    receipts_df[cross_keys + ["value"] + cross_context].rename(columns={"value": "receipts"}),
    on=cross_keys, how="outer", validate="one_to_one", suffixes=("_arrivals", "_receipts"),
    indicator="presence_lignes",
)
expected_pairs = len(pd.concat([arrivals_df[cross_keys], receipts_df[cross_keys]]).drop_duplicates())
assert len(cross_potential) == expected_pairs and not cross_potential.duplicated(cross_keys).any()
cross_complete_mask = cross_potential[["arrivals", "receipts"]].notna().all(axis=1)
cross_incomplete = cross_potential.loc[~cross_complete_mask].copy()
cross_incomplete["mesure_absente"] = [
    "arrivals et receipts" if pd.isna(a) and pd.isna(r) else "arrivals" if pd.isna(a) else "receipts"
    for a, r in zip(cross_incomplete["arrivals"], cross_incomplete["receipts"])
]
cross_panel = cross_potential.loc[cross_complete_mask].copy()
assert not cross_panel.empty and cross_panel["presence_lignes"].eq("both").all()
assert cross_panel["unit_arrivals"].eq("persons").all() and cross_panel["unit_receipts"].eq("current_USD").all()
assert np.isfinite(cross_panel[["arrivals", "receipts"]]).all().all()
assert cross_panel[["arrivals", "receipts"]].ge(0).all().all()
display(pd.DataFrame({
    "Contrôle": ["Couples potentiels (union des clés)", "Couples complets", "Couples incomplets exclus"],
    "Nombre": [len(cross_potential), len(cross_panel), len(cross_incomplete)],
}))
display(cross_incomplete[cross_keys + ["arrivals", "receipts", "mesure_absente", "presence_lignes"]])
cross_coverage = cross_panel.groupby("destination").agg(
    premiere_annee_complete=("year", "min"), derniere_annee_complete=("year", "max"),
    nombre_annees_completes=("year", "nunique"),
).reindex(DESTINATIONS)
display(cross_coverage)
cross_year_sets = [set(cross_panel.loc[cross_panel["destination"].eq(d), "year"]) for d in DESTINATIONS]
cross_common_years = sorted(set.intersection(*cross_year_sets))
assert cross_common_years, "Aucune année commune complète."
cross_first_year, cross_last_year = int(cross_common_years[0]), int(cross_common_years[-1])
display(pd.DataFrame({
    "Première année commune complète": [cross_first_year],
    "Dernière année commune complète": [cross_last_year],
    "Nombre d'années communes complètes": [len(cross_common_years)],
}))
display(Markdown(
    f"**Interprétation calculée.** Sur {len(cross_potential)} couples potentiels, "
    f"{len(cross_panel)} sont complets et {len(cross_incomplete)} exclus pour absence d'au moins une mesure. "
    f"Les sept destinations partagent {len(cross_common_years)} années complètes, "
    f"de {cross_first_year} à {cross_last_year}."
))
cross_charts = []

,Contrôle,Nombre
0,Couples potentiels (union des clés),182
1,Couples complets,177
2,Couples incomplets exclus,5


,destination,year,arrivals,receipts,mesure_absente,presence_lignes
51,Kenya,2020,NaN,NaN,arrivals et receipts,both
104,Tanzanie,1995,295000.0,NaN,receipts,both
105,Tanzanie,1996,326000.0,NaN,receipts,both
129,Tanzanie,2020,NaN,NaN,arrivals et receipts,both
181,Égypte,2020,NaN,4.874000e+09,arrivals,both


,premiere_annee_complete,derniere_annee_complete,nombre_annees_completes
destination,,,
Afrique du Sud,1995,2020,26
Égypte,1995,2019,25
Kenya,1995,2019,25
Maroc,1995,2020,26
Maurice,1995,2020,26
Tanzanie,1997,2019,23
Tunisie,1995,2020,26


,Première année commune complète,Dernière année commune complète,Nombre d'années communes complètes
0,1997,2019,23


**Interprétation calculée.** Sur 182 couples potentiels, 177 sont complets et 5 exclus pour absence d'au moins une mesure. Les sept destinations partagent 23 années complètes, de 1997 à 2019.

**Précaution.** Une paire renseignée est comparable au sens de la disponibilité et de la clé, pas une preuve d'équivalence conceptuelle des définitions. Les métadonnées de chaque série sont conservées séparément. Le panel complet n'est pas nécessairement équilibré entre destinations.

### 6.2 Relation entre arrivées et recettes

**Question :** davantage d'arrivées est-il associé descriptivement à davantage de recettes sur le panel complet ? Chaque point représente une destination et une année. Le coefficient de Pearson résume le panel groupé ; il ne sépare pas les différences entre destinations des tendances temporelles communes. Aucun test causal ou de significativité n'est réalisé.

In [34]:
cross_scatter_data = cross_panel.assign(receipts_billions=cross_panel["receipts"] / 1_000_000_000)
cross_scatter = alt.Chart(cross_scatter_data).mark_circle(size=45, opacity=0.65).encode(
    x=alt.X("arrivals:Q", title="Arrivées internationales (personnes)", axis=alt.Axis(format=",.0f")),
    y=alt.Y("receipts_billions:Q", title="Recettes — milliards USD courants"),
    color=alt.Color("destination:N", title="Destination", scale=arrival_colors, legend=alt.Legend(orient="bottom", columns=3)),
    tooltip=[
        alt.Tooltip("destination:N", title="Destination"),
        alt.Tooltip("year:Q", title="Année", format="d"),
        alt.Tooltip("arrivals:Q", title="Arrivées", format=",.0f"),
        alt.Tooltip("receipts:Q", title="Recettes (USD courants)", format=",.0f"),
    ],
).properties(width=760, height=360, title=alt.Title(
    f"Association descriptive — {int(cross_panel.year.min())}–{int(cross_panel.year.max())}",
    subtitle="Source : WDI ; couples complets, couvertures variables selon la destination.",
))
cross_charts.append(cross_scatter)
display(cross_scatter)
pearson_valid = len(cross_panel) >= 2 and cross_panel[["arrivals", "receipts"]].nunique().gt(1).all()
cross_pearson = cross_panel["arrivals"].corr(cross_panel["receipts"], method="pearson") if pearson_valid else np.nan
display(pd.DataFrame({"Nombre de couples": [len(cross_panel)], "Pearson descriptif": [cross_pearson]}))
association_text = (
    f"Une association descriptive {'positive' if cross_pearson > 0 else 'négative' if cross_pearson < 0 else 'linéaire nulle'} apparaît "
    f"(Pearson = {cross_pearson:.3f}, n = {len(cross_panel)})."
    if pd.notna(cross_pearson) else "La corrélation n'est pas calculable : effectif ou variabilité insuffisant."
)
display(Markdown("**Interprétation calculée.** " + association_text + " Les observations mêlent destinations et années ; ce coefficient ne décrit pas nécessairement la relation au sein de chaque destination."))

alt.Chart(...)

,Nombre de couples,Pearson descriptif
0,177,0.91252


**Interprétation calculée.** Une association descriptive positive apparaît (Pearson = 0.913, n = 177). Les observations mêlent destinations et années ; ce coefficient ne décrit pas nécessairement la relation au sein de chaque destination.

**Précaution.** Association ≠ causalité. Les séries peuvent partager des tendances temporelles et les recettes sont nominales. Le nuage ne démontre pas qu'une augmentation des arrivées entraîne une augmentation des recettes.

### 6.3 Ratio recettes touristiques par arrivée

**Question :** quel ordre de grandeur prend le ratio agrégé `receipts / arrivals` sur les couples complets avec `arrivals > 0` ? Le résultat est en **USD courants par arrivée**, sans correction d'inflation. Les ratios non calculables restent manquants. Les résumés portent sur les ratios annuels, sans pondération par le nombre d'arrivées.

In [35]:
RATIO = "receipts_per_arrival_USD"
cross_ratio = cross_panel.copy()
cross_ratio[RATIO] = np.nan
ratio_valid = cross_ratio["arrivals"].gt(0)
cross_ratio.loc[ratio_valid, RATIO] = cross_ratio.loc[ratio_valid, "receipts"] / cross_ratio.loc[ratio_valid, "arrivals"]
ratio_available = cross_ratio.loc[cross_ratio[RATIO].notna()].copy()
assert not ratio_available.empty
assert ratio_available["arrivals"].gt(0).all()
assert np.isfinite(ratio_available[RATIO]).all() and ratio_available[RATIO].ge(0).all()
assert cross_ratio.loc[~ratio_valid, RATIO].isna().all()
ratio_summary = ratio_available[RATIO].agg(["count", "min", "median", "mean", "max"]).to_frame("USD courants par arrivée (count = effectif)")
display(ratio_summary.style.format("{:,.2f}"))
ratio_by_destination = ratio_available.groupby("destination").agg(
    nombre_observations=(RATIO, "count"), ratio_median=(RATIO, "median"),
    ratio_mean=(RATIO, "mean"), ratio_min=(RATIO, "min"), ratio_max=(RATIO, "max"),
).reindex(DESTINATIONS)
display(ratio_by_destination.style.format({
    "nombre_observations": "{:.0f}", **{c: "{:,.2f}" for c in ["ratio_median", "ratio_mean", "ratio_min", "ratio_max"]},
}))
display(Markdown(
    f"**Interprétation calculée.** {len(ratio_available)} ratios sont calculables ; "
    f"{int((~ratio_valid).sum())} couple complet est exclu du ratio pour dénominateur non positif. "
    f"La médiane du panel est de {ratio_summary.loc['median'].iloc[0]:,.2f} USD courants par arrivée, "
    f"contre une moyenne de {ratio_summary.loc['mean'].iloc[0]:,.2f}. "
    f"Les valeurs vont de {ratio_summary.loc['min'].iloc[0]:,.2f} à {ratio_summary.loc['max'].iloc[0]:,.2f}."
))

,USD courants par arrivée (count = effectif)
count,177.00
min,252.69
median,885.01
mean,948.87
max,"1,879.38"


,nombre_observations,ratio_median,ratio_mean,ratio_min,ratio_max
destination,,,,,
Afrique du Sud,26,680.35,766.06,551.12,"1,147.78"
Égypte,25,909.11,888.66,612.34,"1,119.69"
Kenya,25,880.99,922.73,482.16,"1,356.82"
Maroc,26,774.81,803.05,511.44,"1,610.99"
Maurice,26,"1,486.28","1,477.53","1,079.65","1,879.38"
Tanzanie,23,"1,628.60","1,427.07",744.82,"1,795.61"
Tunisie,26,422.18,408.85,252.69,554.47


**Interprétation calculée.** 177 ratios sont calculables ; 0 couple complet est exclu du ratio pour dénominateur non positif. La médiane du panel est de 885.01 USD courants par arrivée, contre une moyenne de 948.87. Les valeurs vont de 252.69 à 1,879.38.

**Précaution.** Privilégier la médiane pour situer le ratio typique sans donner trop de poids aux extrêmes. La moyenne des ratios annuels n'est pas le ratio des sommes. Les résumés par destination couvrent des périodes variables, documentées en 6.1 : ils ne constituent pas un classement homogène. Un ratio élevé n'est ni une dépense individuelle mesurée ni une preuve de supériorité économique.

### 6.4 Comparaison des destinations sur l'année commune

**Question :** comment les positions changent-elles entre volume d'arrivées, recettes et ratio agrégé ? Utiliser la dernière année commune complète calculée depuis le panel. Les rangs décroissants portent sur cette seule année ; les ex æquo reçoivent le même rang. Le graphique est consacré au ratio, sans répéter les graphiques de niveaux des sections précédentes.

In [36]:
cross_comparison = cross_ratio.loc[cross_ratio["year"].eq(cross_last_year), [
    "destination", "arrivals", "receipts", RATIO,
]].set_index("destination").reindex(DESTINATIONS).reset_index()
assert len(cross_comparison) == len(DESTINATIONS)
assert cross_comparison[["arrivals", "receipts"]].notna().all().all()
for field, rank_name in [("arrivals", "rang_arrivals"), ("receipts", "rang_receipts"), (RATIO, "rang_ratio")]:
    cross_comparison[rank_name] = cross_comparison[field].rank(method="min", ascending=False).astype("Int64")
display(Markdown(f"**Année commune complète : {cross_last_year}. Recettes en USD courants ; ratio en USD courants par arrivée.**"))
display(cross_comparison.style.format({"arrivals": format_count, "receipts": format_count, RATIO: "{:,.2f}"}, na_rep="Non calculable"))
cross_ratio_bar = alt.Chart(cross_comparison.loc[cross_comparison[RATIO].notna()]).mark_bar().encode(
    y=alt.Y("destination:N", title="Destination", sort="-x"),
    x=alt.X(f"{RATIO}:Q", title="Ratio recettes / arrivées — USD courants par arrivée"),
    color=alt.Color("destination:N", scale=arrival_colors, legend=None),
    tooltip=[alt.Tooltip("destination:N", title="Destination"), alt.Tooltip(f"{RATIO}:Q", title="USD courants par arrivée", format=".2f")],
).properties(width=700, height=280, title=alt.Title(
    f"Ratio agrégé sur l'année commune — {cross_last_year}", subtitle="Source : WDI ; ratio descriptif, non corrigé de l'inflation.",
))
cross_charts.append(cross_ratio_bar)
display(cross_ratio_bar)
leaders = {
    rank: ", ".join(cross_comparison.loc[cross_comparison[rank].eq(1), "destination"])
    for rank in ["rang_arrivals", "rang_receipts", "rang_ratio"]
}
rank_changes = cross_comparison.loc[cross_comparison[["rang_arrivals", "rang_receipts", "rang_ratio"]].nunique(axis=1).gt(1)]
rank_descriptions = "; ".join(
    f"{row.destination} : {row.rang_arrivals}/{row.rang_receipts}/{row.rang_ratio}"
    for row in rank_changes.itertuples()
)
display(Markdown(
    f"**Interprétation calculée.** En {cross_last_year}, le premier rang revient à "
    f"**{leaders['rang_arrivals']}** pour les arrivées, **{leaders['rang_receipts']}** pour les recettes "
    f"et **{leaders['rang_ratio']}** pour le ratio. "
    f"Positions différentes (arrivées / recettes / ratio) : {rank_descriptions or 'aucune'}."
))

**Année commune complète : 2019. Recettes en USD courants ; ratio en USD courants par arrivée.**

,destination,arrivals,receipts,receipts_per_arrival_USD,rang_arrivals,rang_receipts,rang_ratio
0,Afrique du Sud,14 797 000,9 064 000 000,612.56,1,3,6
1,Égypte,13 026 000,14 256 000 000,"1,094.43",3,1,3
2,Kenya,2 049 000,1 762 000 000,859.93,5,7,4
3,Maroc,13 109 000,9 949 000 000,758.94,2,2,5
4,Maurice,1 418 000,2 024 000 000,"1,427.36",7,6,2
5,Tanzanie,1 527 000,2 624 500 000,"1,718.73",6,5,1
6,Tunisie,9 429 000,2 683 000 000,284.55,4,4,7


alt.Chart(...)

**Interprétation calculée.** En 2019, le premier rang revient à **Afrique du Sud** pour les arrivées, **Égypte** pour les recettes et **Tanzanie** pour le ratio. Positions différentes (arrivées / recettes / ratio) : Afrique du Sud : 1/3/6; Égypte : 3/1/3; Kenya : 5/7/4; Maroc : 2/2/5; Maurice : 7/6/2; Tanzanie : 6/5/1; Tunisie : 4/4/7.

**Précaution.** Le rang dépend de l'indicateur choisi. Le ratio distingue le montant de recettes agrégées rapporté au nombre d'arrivées ; il ne contient aucune information sur les coûts et ne mesure donc pas une marge économique.

### 6.5 Évolution du ratio dans le temps

**Question :** comment le ratio nominal évolue-t-il sur les années où il est calculable ? Les segments suivent les années consécutives de chaque destination. Un trou n'est jamais relié artificiellement. Les bornes renseignées sont affichées pour étayer la lecture ; aucune trajectoire intermédiaire n'est reconstruite.

In [37]:
# Réutilisation de la segmentation existante via une copie de présentation.
cross_ratio_plot = segment_arrivals(ratio_available[["destination", "year", RATIO]].rename(columns={RATIO: "value"})).rename(columns={"value": RATIO})
cross_ratio_line = alt.Chart(cross_ratio_plot).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Année", axis=alt.Axis(format="d", tickMinStep=1)),
    y=alt.Y(f"{RATIO}:Q", title="Recettes touristiques par arrivée — USD courants"),
    color=alt.Color("destination:N", title="Destination", scale=arrival_colors, legend=alt.Legend(orient="bottom", columns=3)),
    detail=alt.Detail("segment:N"), order=alt.Order("year:Q"),
    tooltip=[alt.Tooltip("destination:N", title="Destination"), alt.Tooltip("year:Q", title="Année", format="d"), alt.Tooltip(f"{RATIO}:Q", title="USD courants par arrivée", format=".2f")],
).properties(width=760, height=360, title=alt.Title(
    f"Évolution du ratio nominal — {int(ratio_available.year.min())}–{int(ratio_available.year.max())}",
    subtitle="Source : WDI ; couvertures propres aux destinations, aucune interpolation.",
))
cross_charts.append(cross_ratio_line)
display(cross_ratio_line)
ratio_ordered = ratio_available.sort_values(["destination", "year"])
ratio_endpoints = ratio_ordered.groupby("destination").agg(
    annee_debut=("year", "first"), annee_fin=("year", "last"),
    ratio_debut=(RATIO, "first"), ratio_fin=(RATIO, "last"),
).reindex(DESTINATIONS)
display(ratio_endpoints.style.format({"ratio_debut": "{:,.2f}", "ratio_fin": "{:,.2f}"}))
ratio_up = int(ratio_endpoints["ratio_fin"].gt(ratio_endpoints["ratio_debut"]).sum())
ratio_down = int(ratio_endpoints["ratio_fin"].lt(ratio_endpoints["ratio_debut"]).sum())
ratio_equal = int(ratio_endpoints["ratio_fin"].eq(ratio_endpoints["ratio_debut"]).sum())
display(Markdown(
    f"**Interprétation calculée.** Entre leurs bornes propres, {ratio_up} destinations présentent "
    f"un ratio final supérieur au ratio initial, {ratio_down} un ratio inférieur et {ratio_equal} un ratio identique. "
    "Les périodes diffèrent : ce constat ne compare pas des taux de croissance homogènes et ne signifie pas une progression régulière."
))

alt.Chart(...)

,annee_debut,annee_fin,ratio_debut,ratio_fin
destination,,,,
Afrique du Sud,1995,2020,566.61,698.81
Égypte,1995,2019,942.87,"1,094.43"
Kenya,1995,2019,805.95,859.93
Maroc,1995,2020,533.79,"1,610.99"
Maurice,1995,2020,"1,409.61","1,639.24"
Tanzanie,1997,2019,952.78,"1,718.73"
Tunisie,1995,2020,446.12,500.50


**Interprétation calculée.** Entre leurs bornes propres, 7 destinations présentent un ratio final supérieur au ratio initial, 0 un ratio inférieur et 0 un ratio identique. Les périodes diffèrent : ce constat ne compare pas des taux de croissance homogènes et ne signifie pas une progression régulière.

**Précaution.** Une hausse du ratio peut notamment refléter l'évolution nominale des recettes ou celle du dénominateur. Elle ne démontre ni une hausse réelle des dépenses individuelles, ni une amélioration de la qualité touristique, du pouvoir d'achat ou du positionnement de l'offre.

### 6.6 Limites d'interprétation

- Recettes et ratios en **USD courants**, sans correction d'inflation.
- Ratio de séries agrégées, pas une mesure individuelle ; les définitions statistiques du numérateur et du dénominateur doivent être vérifiées avant toute lecture économique avancée.
- Couverture temporelle variable et couples manquants explicitement exclus, sans remplacement par zéro.
- Corrélation groupée sensible aux différences entre destinations et aux tendances temporelles partagées ; **association ≠ causalité**.
- Une année commune améliore la cohérence temporelle sans résoudre toutes les différences de définition.
- La couverture post-2020 est contrôlée ci-dessous ; aucune reconstruction à partir des provenances n'est autorisée.

**Contrôle final :** vérifier les données sources, les clés, les ratios et les trois graphiques.

In [38]:
cross_post_count = int(cross_panel["year"].gt(2020).sum())
display(Markdown(
    f"**Couverture post-2020 recalculée : {cross_post_count} couple complet.** " +
    ("La reprise nationale post-Covid ne peut pas être étudiée correctement ; un enrichissement des séries nationales serait nécessaire."
     if cross_post_count == 0 else "La couverture a évolué : examiner son étendue avant toute analyse de reprise.")
))
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
pd.testing.assert_frame_equal(arrivals_df, cross_arrivals_initial, check_exact=True)
pd.testing.assert_frame_equal(receipts_df, cross_receipts_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
assert len(cross_panel) + len(cross_incomplete) == len(cross_potential) == expected_pairs
assert not cross_panel.duplicated(cross_keys).any()
assert cross_panel[["arrivals", "receipts"]].notna().all().all()
assert ratio_available["arrivals"].gt(0).all()
np.testing.assert_allclose(ratio_available[RATIO], ratio_available["receipts"] / ratio_available["arrivals"], rtol=0, atol=0)
assert not np.isinf(cross_ratio[RATIO]).any()
assert ratio_available[RATIO].ge(0).all()
assert cross_common_years == sorted(cross_panel.groupby("year")["destination"].nunique().loc[lambda s: s.eq(len(DESTINATIONS))].index.tolist())
assert cross_ratio_plot.groupby(["destination", "segment"])["year"].diff().dropna().eq(1).all()
assert len(cross_charts) == 3
for chart in cross_charts:
    cross_spec = chart.to_dict(validate=True)
    assert cross_spec.get("datasets") and all(len(rows) > 0 for rows in cross_spec["datasets"].values())
print("Section 06 validée : jointure un à un, ratios reproductibles, données sources intactes, trois graphiques non vides.")

**Couverture post-2020 recalculée : 0 couple complet.** La reprise nationale post-Covid ne peut pas être étudiée correctement ; un enrichissement des séries nationales serait nécessaire.

Section 06 validée : jointure un à un, ratios reproductibles, données sources intactes, trois graphiques non vides.


**Fin de la section 06.** Les résultats décrivent les couples nationaux renseignés et leur ratio agrégé. Aucune conclusion causale ou mesure de profit n'est produite ; les étapes suivantes restent hors de cette phase.

## 07 — Dynamiques et croissance

### 7.1 Variations annuelles et disponibilité

La variation est `100 × (valeur_t / valeur_t−1 − 1)`, uniquement pour deux années consécutives renseignées et une base strictement positive. Les couches nationales sont analysées séparément : aucune interpolation, aucun remplacement des absences. Les CAGR des sections précédentes ne sont pas recalculés.

In [39]:
growth_sources = {"Arrivées": arrivals_df, "Recettes": receipts_df}
growth_snapshots = {k: v.copy(deep=True) for k, v in growth_sources.items()}
def annual_changes(table):
    result = table.sort_values(["destination", "year"]).copy()
    assert not result.duplicated(["destination", "year"]).any()
    grouped = result.groupby("destination")
    result["previous_year"] = grouped["year"].shift()
    result["previous_value"] = grouped["value"].shift()
    valid = result["year"].sub(result["previous_year"]).eq(1) & result["value"].notna() & result["previous_value"].gt(0)
    result["variation_pct"] = np.nan
    result.loc[valid, "variation_pct"] = 100 * (result.loc[valid, "value"] / result.loc[valid, "previous_value"] - 1)
    result["statut"] = np.select(
        [result["previous_year"].isna(), result["year"].sub(result["previous_year"]).ne(1),
         result["value"].isna() | result["previous_value"].isna(), result["previous_value"].le(0)],
        ["Première observation", "Années non consécutives", "Valeur manquante", "Base non positive"],
        default="Calculable")
    assert result["variation_pct"].notna().eq(valid).all()
    return result
growth_all = pd.concat([annual_changes(v).assign(indicateur=k) for k, v in growth_sources.items()], ignore_index=True)
display(growth_all.groupby(["indicateur", "statut"]).size().rename("nombre").to_frame())
display(growth_all[["indicateur", "destination", "year", "variation_pct", "statut"]])
growth_pre = growth_all.loc[growth_all["year"].lt(2020) & growth_all["variation_pct"].notna()].copy()
growth_common_years = sorted(growth_pre.groupby("year").size().loc[lambda s: s.eq(2 * len(DESTINATIONS))].index)
assert growth_common_years
growth_comparable = growth_pre.loc[growth_pre["year"].isin(growth_common_years)].copy()
display(Markdown(f"**Lecture.** {len(growth_pre)} variations sont disponibles avant 2020. La comparaison homogène utilise {len(growth_common_years)} années communes aux deux indicateurs et aux sept destinations, de {min(growth_common_years)} à {max(growth_common_years)} (années de fin des variations)."))

nombre
indicateur statut                      
Arrivées   Calculable               172
           Première observation       7
           Valeur manquante           3
Recettes   Calculable               171
           Première observation       7
           Valeur manquante           4

,indicateur,destination,year,variation_pct,statut
0,Arrivées,Afrique du Sud,1995,NaN,Première observation
1,Arrivées,Afrique du Sud,1996,10.717336,Calculable
2,Arrivées,Afrique du Sud,1997,-0.308523,Calculable
3,Arrivées,Afrique du Sud,1998,14.081238,Calculable
4,Arrivées,Afrique du Sud,1999,2.170227,Calculable
...,...,...,...,...,...
359,Recettes,Égypte,2016,-52.066116,Calculable
360,Recettes,Égypte,2017,161.222021,Calculable
361,Recettes,Égypte,2018,47.105141,Calculable
362,Recettes,Égypte,2019,12.216625,Calculable


**Lecture.** 334 variations sont disponibles avant 2020. La comparaison homogène utilise 22 années communes aux deux indicateurs et aux sept destinations, de 1998 à 2019 (années de fin des variations).

### 7.2 Comparaison avant 2020 et volatilité

La comparaison utilise les mêmes années pour les sept destinations et les deux indicateurs. La volatilité est l'écart-type échantillonnal des variations annuelles, en points de pourcentage ; elle décrit leur dispersion, sans mesurer un risque futur. Les recettes restent nominales, en USD courants.

In [40]:
growth_summary = growth_comparable.groupby(["indicateur", "destination"]).agg(
    nombre=("variation_pct", "count"), moyenne_pct=("variation_pct", "mean"),
    mediane_pct=("variation_pct", "median"), volatilite_points=("variation_pct", "std"),
    minimum_pct=("variation_pct", "min"), maximum_pct=("variation_pct", "max"))
display(growth_summary.round(2))
growth_heatmap = alt.Chart(growth_comparable).mark_rect().encode(
    x=alt.X("year:O", title="Année de fin"),
    y=alt.Y("destination:N", title=None, sort=DESTINATIONS),
    color=alt.Color("variation_pct:Q", title="Variation (%)", scale=alt.Scale(scheme="redblue", domainMid=0)),
    tooltip=["indicateur:N", "destination:N", "year:O", alt.Tooltip("variation_pct:Q", format=".2f")]
).properties(width=720, height=190).facet(row=alt.Row("indicateur:N", title=None))
display(growth_heatmap)
for growth_label in growth_sources:
    growth_part = growth_summary.loc[growth_label]
    growth_max_vol = growth_part["volatilite_points"].max()
    growth_min_vol = growth_part["volatilite_points"].min()
    display(Markdown(
        f"**{growth_label}.** Dispersion maximale : {', '.join(growth_part.index[growth_part.volatilite_points.eq(growth_max_vol)])} "
        f"({growth_max_vol:.2f} points) ; minimale : {', '.join(growth_part.index[growth_part.volatilite_points.eq(growth_min_vol)])} "
        f"({growth_min_vol:.2f} points), sur la période commune."))

nombre  moyenne_pct  mediane_pct  \
indicateur destination                                        
Arrivées   Afrique du Sud      22         5.07         4.15   
           Kenya               22         4.45         8.06   
           Maroc               22         6.77         5.76   
           Maurice             22         4.45         3.86   
           Tanzanie            22         7.39         5.10   
           Tunisie             22         4.45         4.86   
           Égypte              22         8.19        13.24   
Recettes   Afrique du Sud      22         5.76         0.50   
           Kenya               22         4.84         6.73   
           Maroc               22         9.12         8.78   
           Maurice             22         5.75         7.85   
           Tanzanie            22        10.66         8.48   
           Tunisie             22         3.01         5.97   
           Égypte              22        12.22        13.34   

                           volatilite_points  minimum_pct  maximum_pct  
indicateur destination                                                  
Arrivées   Afrique du Sud               6.15        -3.98        18.58  
           Kenya                       15.32       -33.79        39.75  
           Maroc                        5.96         0.33        25.13  
           Maurice                      5.08        -8.25        15.61  
           Tanzanie                    11.62       -20.10        33.89  
           Tunisie                     12.35       -26.60        23.20  
           Égypte                      23.23       -42.12        53.58  
Recettes   Afrique du Sud              18.78       -12.82        80.62  
           Kenya                       21.03       -58.71        44.13  
           Maroc                       11.65       -14.39        30.09  
           Maurice                     10.92       -23.75        27.73  
           Tanzanie                    15.71       -18.42        64.30  
           Tunisie                     15.94       -38.56        30.19  
           Égypte                      42.12       -52.07       161.22

alt.FacetChart(...)

**Arrivées.** Dispersion maximale : Égypte (23.23 points) ; minimale : Maurice (5.08 points), sur la période commune.

**Recettes.** Dispersion maximale : Égypte (42.12 points) ; minimale : Maurice (10.92 points), sur la période commune.

### 7.3 Années de forte hausse ou baisse

Les extrêmes désignent ici le minimum et le maximum observés avant 2020 pour chaque destination et indicateur, sans seuil arbitraire ni attribution causale. Ce tableau utilise toute la couverture disponible, explicitée par ses bornes ; ses périodes peuvent donc différer de la comparaison homogène.

In [41]:
growth_groups = growth_pre.groupby(["indicateur", "destination"])["variation_pct"]
growth_extremes = growth_pre.loc[
    growth_pre["variation_pct"].eq(growth_groups.transform("min")) |
    growth_pre["variation_pct"].eq(growth_groups.transform("max")),
    ["indicateur", "destination", "year", "variation_pct"]
].copy()
growth_coverage = growth_pre.groupby(["indicateur", "destination"]).agg(
    premiere_variation=("year", "min"), derniere_variation=("year", "max"), nombre=("year", "size"))
display(growth_coverage)
display(growth_extremes.sort_values(["indicateur", "destination", "variation_pct"]).round(2))
for growth_label, growth_part in growth_pre.groupby("indicateur"):
    for growth_kind, growth_value in [("minimum", growth_part.variation_pct.min()), ("maximum", growth_part.variation_pct.max())]:
        growth_rows = growth_part.loc[growth_part.variation_pct.eq(growth_value)]
        growth_names = "; ".join(f"{r.destination}, {r.year}" for r in growth_rows.itertuples())
        display(Markdown(f"**{growth_label} — {growth_kind} avant 2020 : {growth_value:.2f} %**, observé pour {growth_names}."))

premiere_variation  derniere_variation  nombre
indicateur destination                                                   
Arrivées   Afrique du Sud                1996                2019      24
           Kenya                         1996                2019      24
           Maroc                         1996                2019      24
           Maurice                       1996                2019      24
           Tanzanie                      1996                2019      24
           Tunisie                       1996                2019      24
           Égypte                        1996                2019      24
Recettes   Afrique du Sud                1996                2019      24
           Kenya                         1996                2019      24
           Maroc                         1996                2019      24
           Maurice                       1996                2019      24
           Tanzanie                      1998                2019      22
           Tunisie                       1996                2019      24
           Égypte                        1996                2019      24

,indicateur,destination,year,variation_pct
20,Arrivées,Afrique du Sud,2015,-3.98
15,Arrivées,Afrique du Sud,2010,18.58
39,Arrivées,Kenya,2008,-33.79
49,Arrivées,Kenya,2018,39.75
68,Arrivées,Maroc,2011,0.33
56,Arrivées,Maroc,1999,25.13
92,Arrivées,Maurice,2009,-8.25
79,Arrivées,Maurice,1996,16.48
109,Arrivées,Tanzanie,2000,-20.10
107,Arrivées,Tanzanie,1998,33.89


**Arrivées — minimum avant 2020 : -42.12 %**, observé pour Égypte, 2016.

**Arrivées — maximum avant 2020 : 53.58 %**, observé pour Égypte, 2017.

**Recettes — minimum avant 2020 : -58.71 %**, observé pour Kenya, 2000.

**Recettes — maximum avant 2020 : 161.22 %**, observé pour Égypte, 2017.

### 7.4 Rupture exceptionnelle de 2020 et limites

2020 est exclue des statistiques précédentes et présentée séparément. Seules les variations 2019–2020 calculables sont tracées ; les absences restent visibles dans le tableau. Les évolutions des recettes ne constituent pas des évolutions réelles corrigées de l'inflation. Aucun mécanisme causal n'est estimé.

In [42]:
growth_break = growth_all.loc[growth_all["year"].eq(2020), ["indicateur", "destination", "year", "variation_pct", "statut"]].copy()
display(growth_break.round(2))
growth_break_valid = growth_break.loc[growth_break["variation_pct"].notna()]
growth_break_chart = alt.Chart(growth_break_valid).mark_bar().encode(
    x=alt.X("variation_pct:Q", title="Variation 2019–2020 (%)"),
    y=alt.Y("destination:N", title=None, sort=DESTINATIONS),
    color=alt.Color("indicateur:N", title="Indicateur"),
    tooltip=["destination:N", "indicateur:N", alt.Tooltip("variation_pct:Q", format=".2f")]
).properties(width=320, height=220).facet(column=alt.Column("indicateur:N", title=None))
display(growth_break_chart)
for growth_label, growth_part in growth_break.groupby("indicateur"):
    growth_valid_part = growth_part.dropna(subset=["variation_pct"])
    display(Markdown(f"**2020 — {growth_label} :** {len(growth_valid_part)} variations calculables, "
        f"{len(growth_part) - len(growth_valid_part)} absentes ; "
        f"de {growth_valid_part.variation_pct.min():.2f} % à {growth_valid_part.variation_pct.max():.2f} %."))
growth_post_count = int(growth_all.loc[growth_all.year.gt(2020), "value"].notna().sum())
display(Markdown(f"**Après 2020 : {growth_post_count} observations nationales renseignées.** "
    + ("La couverture ne permet pas de calculer une reprise post-Covid." if growth_post_count == 0
       else "Une vérification de la couverture serait nécessaire avant toute estimation de reprise.")))

,indicateur,destination,year,variation_pct,statut
25,Arrivées,Afrique du Sud,2020,-73.73,Calculable
51,Arrivées,Kenya,2020,NaN,Valeur manquante
77,Arrivées,Maroc,2020,-78.63,Calculable
103,Arrivées,Maurice,2020,-77.72,Calculable
129,Arrivées,Tanzanie,2020,NaN,Valeur manquante
155,Arrivées,Tunisie,2020,-78.66,Calculable
181,Arrivées,Égypte,2020,NaN,Valeur manquante
207,Recettes,Afrique du Sud,2020,-70.04,Calculable
233,Recettes,Kenya,2020,NaN,Valeur manquante
259,Recettes,Maroc,2020,-54.63,Calculable


alt.FacetChart(...)

**2020 — Arrivées :** 4 variations calculables, 3 absentes ; de -78.66 % à -73.73 %.

**2020 — Recettes :** 5 variations calculables, 2 absentes ; de -74.41 % à -54.63 %.

**Après 2020 : 0 observations nationales renseignées.** La couverture ne permet pas de calculer une reprise post-Covid.

### 7.5 Contrôles

Les calculs sont vérifiés contre un appariement explicite avec l'année précédente. Un contrôle sur une copie privée retire une année réellement observée pour vérifier qu'aucune variation ne franchit ce trou. Aucune donnée de contrôle n'entre dans l'analyse.

In [43]:
for growth_label, growth_source in growth_sources.items():
    pd.testing.assert_frame_equal(growth_source, growth_snapshots[growth_label], check_exact=True)
    growth_actual = growth_all.loc[growth_all.indicateur.eq(growth_label)]
    growth_prior = growth_source[["destination", "year", "value"]].copy()
    growth_prior["year"] += 1
    growth_reference = growth_source[["destination", "year", "value"]].merge(
        growth_prior.rename(columns={"value": "base"}), on=["destination", "year"],
        how="left", validate="one_to_one")
    growth_reference = growth_reference.loc[growth_reference.value.notna() & growth_reference.base.gt(0)]
    growth_check = growth_actual.loc[growth_actual.variation_pct.notna()].merge(
        growth_reference, on=["destination", "year"], validate="one_to_one", suffixes=("", "_reference"))
    assert len(growth_check) == len(growth_reference) == growth_actual.variation_pct.notna().sum()
    np.testing.assert_allclose(growth_check.variation_pct, 100 * (growth_check.value_reference / growth_check.base - 1), rtol=0, atol=0)
    growth_probe = growth_source.loc[growth_source.destination.eq(DESTINATIONS[0])].sort_values("year")
    growth_removed_year = int(growth_probe.year.iloc[2])
    growth_probe_result = annual_changes(growth_probe.loc[growth_probe.year.ne(growth_removed_year)])
    assert growth_probe_result.loc[growth_probe_result.year.eq(growth_removed_year + 1), "variation_pct"].isna().all()
assert not growth_all.duplicated(["indicateur", "destination", "year"]).any()
assert np.isfinite(growth_all.variation_pct.dropna()).all()
assert growth_all.loc[growth_all.variation_pct.notna(), "previous_value"].gt(0).all()
assert growth_comparable.year.lt(2020).all()
assert growth_comparable.groupby(["indicateur", "destination"]).size().eq(len(growth_common_years)).all()
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
growth_charts = [growth_heatmap, growth_break_chart]
for growth_chart in growth_charts:
    growth_spec = growth_chart.to_dict(validate=True)
    assert growth_spec.get("datasets") and all(growth_spec["datasets"].values())
print("Section 07 : contrôles réussis, deux graphiques non vides, sources intactes.")


Section 07 : contrôles réussis, deux graphiques non vides, sources intactes.


## 08 — Provenance des visiteurs

### 8.1 Couverture et unités de comparaison

La couverture est décrite séparément pour chaque combinaison de granularité, type de mesure, unité, périmètre et qualité. Une valeur absente reste absente. Les libellés des pays et territoires suivent la source ; ils ne sont pas harmonisés artificiellement. Aucun classement entre destinations ni rapprochement avec les totaux nationaux de périodes différentes n'est effectué.

In [4]:
if "provenance_df" not in globals():
    provenance_df = df.loc[df["dataset_layer"].eq("provenance")].copy()
prov_initial = provenance_df.copy(deep=True)
prov_df_initial = df.copy(deep=True)
prov_dimensions = ["destination", "granularity", "metric_type", "unit", "coverage_scope", "quality_flag"]
prov_keys = ["destination", "year", "origin_name", "granularity", "metric_type"]
assert provenance_df.dataset_layer.eq("provenance").all()
assert not provenance_df.duplicated(prov_keys).any()
assert provenance_df[prov_dimensions].notna().all().all()
prov_coverage = provenance_df.groupby(prov_dimensions, dropna=False).agg(
    annees=("year", lambda s: ", ".join(map(str, sorted(s.unique())))),
    lignes=("value", "size"), renseignees=("value", "count"),
    absentes=("value", lambda s: int(s.isna().sum())),
    origines=("origin_name", "nunique")).reset_index()
display(prov_coverage)
prov_missing = provenance_df.loc[provenance_df.value.isna()]
display(prov_missing.groupby(["destination", "year", "quality_flag"]).size().rename("absences").to_frame())
display(Markdown(f"**Couverture calculée :** {len(provenance_df)} lignes, "
    f"{provenance_df.value.notna().sum()} valeurs renseignées et {len(prov_missing)} absences, "
    f"pour {provenance_df.destination.nunique()} destinations. Les lignes absentes ne sont ni supprimées de ce bilan ni remplacées par zéro."))

,destination,granularity,metric_type,unit,coverage_scope,quality_flag,annees,lignes,renseignees,absentes,origines
0,Afrique du Sud,country,tourist_arrivals,persons,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18,"2022, 2023, 2024",54,54,0,18
1,Kenya,country,tourist_arrivals,persons,Top 30 marchés sources publiés,exact_top30,"2022, 2023, 2024",89,89,0,31
2,Kenya,institutional_category,tourist_arrivals,persons,Top 30 marchés sources publiés,exact_top30,2022,1,1,0,1
3,Maroc,aggregate_total,tourist_arrivals,persons,Série officielle incluant agrégats publiés,exact_aggregate,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019...",9,9,0,1
4,Maroc,country,tourist_arrivals,persons,Nationalités/pays publiés par Open Data Maroc,exact_country,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019...",81,81,0,9
5,Maroc,diaspora,diaspora_arrivals,persons,MRE séparés des touristes étrangers,exact_diaspora,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019...",9,9,0,1
6,Maroc,regional_aggregate,tourist_arrivals,persons,Série officielle incluant agrégats publiés,exact_aggregate,"2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019...",18,18,0,2
7,Maurice,country,tourist_arrivals,persons,7 principaux marchés publiés dans les annual h...,exact_main7,"2022, 2023, 2024",21,21,0,7
8,Tanzanie,country,source_market_share,share,Top 15 marchés — parts issues de l'Exit Survey,survey_share_top15,"2022, 2023, 2024",45,45,0,19
9,Tunisie,country,tourist_arrivals,persons,Nationalités publiées par ONTT,exact_country,"2017, 2018, 2019, 2020, 2021, 2022, 2023",297,297,0,51


absences
destination year quality_flag                
Tunisie     2017 missing_unverified        30
            2018 missing_unverified        30

**Couverture calculée :** 716 lignes, 656 valeurs renseignées et 60 absences, pour 7 destinations. Les lignes absentes ne sont ni supprimées de ce bilan ni remplacées par zéro.

### 8.2 Principaux marchés pays dans leur périmètre publié

Le tableau retient jusqu'aux cinq premiers pays/territoires, ex æquo inclus, à la dernière année disponible de chaque périmètre comparable. Les rangs sont internes à ce périmètre et portent uniquement sur les valeurs renseignées. Les agrégats, diasporas et catégories institutionnelles sont exclus de ces rangs.

Afrique du Sud : panel harmonisé de 18 marchés ; Kenya : Top 30 publié, dont la composition peut varier ; Maurice : sept marchés publiés. Maroc et Tunisie : nationalités publiées, sans supposer l'exhaustivité. Les absences tunisiennes empêchent une lecture exhaustive des premières années. Aucun total de panel n'est assimilé au total national. L'Égypte est exclue de tout classement de marchés pays, car un seul pays est vérifié.

In [5]:
prov_volume_flags = ["exact_panel18", "exact_top30", "exact_main7", "exact_country"]
prov_country = provenance_df.loc[
    provenance_df.granularity.eq("country") & provenance_df.metric_type.eq("tourist_arrivals")
    & provenance_df.unit.eq("persons") & provenance_df.quality_flag.isin(prov_volume_flags)
    & provenance_df.value.notna()].copy()
prov_group = prov_dimensions
prov_latest = prov_country.loc[prov_country.year.eq(prov_country.groupby(prov_group).year.transform("max"))].copy()
prov_latest["rang_dans_perimetre"] = prov_latest.groupby(prov_group + ["year"]).value.rank(method="min", ascending=False)
prov_top = prov_latest.loc[prov_latest.rang_dans_perimetre.le(5)].sort_values(prov_group + ["rang_dans_perimetre"])
display(prov_top[["destination", "year", "origin_name", "value", "unit", "rang_dans_perimetre", "coverage_scope", "quality_flag"]])
for prov_identity, prov_part in prov_latest.groupby(prov_group + ["year"]):
    prov_leaders = prov_part.loc[prov_part.rang_dans_perimetre.eq(1)]
    prov_description = "; ".join(f"{r.origin_name} : {r.value:,.0f}" for r in prov_leaders.itertuples())
    display(Markdown(f"**{prov_identity[0]}, {prov_identity[-1]} :** {prov_description} personnes, "
        f"premier rang parmi {len(prov_part)} marchés pays/territoires renseignés dans ce périmètre uniquement."))

,destination,year,origin_name,value,unit,rang_dans_perimetre,coverage_scope,quality_flag
1044,Afrique du Sud,2024,Zimbabwe,2183260.0,persons,1.0,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18
1045,Afrique du Sud,2024,Mozambique,1591751.0,persons,2.0,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18
1046,Afrique du Sud,2024,Lesotho,974369.0,persons,3.0,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18
1047,Afrique du Sud,2024,Eswatini,842318.0,persons,4.0,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18
1048,Afrique du Sud,2024,Botswana,395965.0,persons,5.0,Panel harmonisé: 10 marchés SADC + 8 marchés o...,exact_panel18
912,Kenya,2024,United States of America,306501.0,persons,1.0,Top 30 marchés sources publiés,exact_top30
913,Kenya,2024,Uganda,225559.0,persons,2.0,Top 30 marchés sources publiés,exact_top30
914,Kenya,2024,United Republic of Tanzania,203290.0,persons,3.0,Top 30 marchés sources publiés,exact_top30
915,Kenya,2024,United Kingdom,180639.0,persons,4.0,Top 30 marchés sources publiés,exact_top30
916,Kenya,2024,India,131570.0,persons,5.0,Top 30 marchés sources publiés,exact_top30


**Afrique du Sud, 2024 :** Zimbabwe : 2,183,260 personnes, premier rang parmi 18 marchés pays/territoires renseignés dans ce périmètre uniquement.

**Kenya, 2024 :** United States of America : 306,501 personnes, premier rang parmi 30 marchés pays/territoires renseignés dans ce périmètre uniquement.

**Maroc, 2020 :** France : 412,179 personnes, premier rang parmi 9 marchés pays/territoires renseignés dans ce périmètre uniquement.

**Maurice, 2024 :** France : 339,421 personnes, premier rang parmi 7 marchés pays/territoires renseignés dans ce périmètre uniquement.

**Tunisie, 2023 :** Algériens : 3,038,002 personnes, premier rang parmi 51 marchés pays/territoires renseignés dans ce périmètre uniquement.

### 8.3 Agrégats, diasporas, catégories institutionnelles et Égypte

Les totaux agrégés du Maroc ne sont pas additionnés à leurs composantes. Maghreb, Scandinavie et Scandinaves restent des agrégats régionaux ; MRE et TRE restent des diasporas. La catégorie United Nations Organization au Kenya est institutionnelle, et non un pays.

Pour l'Égypte, la série États-Unis documente un marché isolé, sans permettre d'identifier les principaux marchés pays. Les parts régionales de touristes et de nuitées constituent deux mesures différentes : aucune conversion en volumes, addition entre mesures ou extrapolation aux années absentes.

In [6]:
prov_special = provenance_df.loc[provenance_df.granularity.ne("country")].copy()
display(prov_special.groupby(["destination", "granularity", "metric_type", "unit", "origin_name", "coverage_scope"], dropna=False).agg(
    premiere_annee=("year", "min"), derniere_annee=("year", "max"), lignes=("value", "size"), renseignements=("value", "count")).reset_index())
prov_institution = prov_special.loc[prov_special.granularity.eq("institutional_category")]
display(prov_institution[["destination", "year", "origin_name", "value", "unit", "quality_flag"]])
prov_egypt = provenance_df.loc[provenance_df.quality_flag.isin(["exact_single_country", "regional_share_only"])].copy()
display(prov_egypt[["destination", "year", "origin_name", "granularity", "metric_type", "value", "unit", "quality_flag"]])
prov_egypt_shares = prov_egypt.loc[prov_egypt.unit.eq("share")].copy()
prov_egypt_shares["part_pct"] = 100 * prov_egypt_shares.value
display(prov_egypt_shares[["year", "origin_name", "metric_type", "part_pct"]])
display(Markdown(f"**Limite égyptienne calculée :** "
    f"{prov_egypt.loc[prov_egypt.granularity.eq('country'), 'origin_name'].nunique()} marché pays vérifié ; "
    f"{len(prov_egypt_shares)} observations de parts régionales, à conserver séparées selon leur type de mesure."))

,destination,granularity,metric_type,unit,origin_name,coverage_scope,premiere_annee,derniere_annee,lignes,renseignements
0,Kenya,institutional_category,tourist_arrivals,persons,United Nations Organization,Top 30 marchés sources publiés,2022,2022,1,1
1,Maroc,aggregate_total,tourist_arrivals,persons,Touristes Etrangers,Série officielle incluant agrégats publiés,2012,2020,9,9
2,Maroc,diaspora,diaspora_arrivals,persons,MRE,MRE séparés des touristes étrangers,2012,2020,9,9
3,Maroc,regional_aggregate,tourist_arrivals,persons,Maghreb,Série officielle incluant agrégats publiés,2012,2020,9,9
4,Maroc,regional_aggregate,tourist_arrivals,persons,Scandinavie,Série officielle incluant agrégats publiés,2012,2020,9,9
5,Tunisie,diaspora,diaspora_arrivals,persons,Tunisiens résidents à l'étranger (TRE),Nationalités publiées + TRE séparés,2017,2023,7,7
6,Tunisie,regional_aggregate,tourist_arrivals,persons,Scandinaves,Nationalités publiées par ONTT,2017,2023,7,7
7,Égypte,regional_aggregate,regional_tourist_nights_share,share,Americans,Répartition régionale 2019 uniquement,2019,2019,1,1
8,Égypte,regional_aggregate,regional_tourist_nights_share,share,Arabs,Répartition régionale 2019 uniquement,2019,2019,1,1
9,Égypte,regional_aggregate,regional_tourist_nights_share,share,Europeans,Répartition régionale 2019 uniquement,2019,2019,1,1


,destination,year,origin_name,value,unit,quality_flag
873,Kenya,2022,United Nations Organization,14892.0,persons,exact_top30


,destination,year,origin_name,granularity,metric_type,value,unit,quality_flag
1062,Égypte,2010,United States,country,tourist_arrivals,361533.000,persons,exact_single_country
1063,Égypte,2011,United States,country,tourist_arrivals,184608.000,persons,exact_single_country
1064,Égypte,2012,United States,country,tourist_arrivals,179134.000,persons,exact_single_country
1065,Égypte,2013,United States,country,tourist_arrivals,147624.000,persons,exact_single_country
1066,Égypte,2014,United States,country,tourist_arrivals,154619.000,persons,exact_single_country
1067,Égypte,2015,United States,country,tourist_arrivals,188712.000,persons,exact_single_country
1068,Égypte,2016,United States,country,tourist_arrivals,184341.000,persons,exact_single_country
1069,Égypte,2017,United States,country,tourist_arrivals,226429.000,persons,exact_single_country
1070,Égypte,2018,United States,country,tourist_arrivals,287796.000,persons,exact_single_country
1071,Égypte,2019,United States,country,tourist_arrivals,349596.000,persons,exact_single_country


,year,origin_name,metric_type,part_pct
1072,2019,Europeans,regional_tourist_share,64.3
1073,2019,Arabs,regional_tourist_share,24.3
1074,2019,Americans,regional_tourist_share,4.2
1075,2019,Others,regional_tourist_share,7.2
1076,2019,Europeans,regional_tourist_nights_share,53.5
1077,2019,Arabs,regional_tourist_nights_share,34.5
1078,2019,Americans,regional_tourist_nights_share,5.4
1079,2019,Others,regional_tourist_nights_share,6.6


**Limite égyptienne calculée :** 1 marché pays vérifié ; 8 observations de parts régionales, à conserver séparées selon leur type de mesure.

### 8.4 Tanzanie : parts des marchés dans l'enquête

Les parts du Top 15 de l'Exit Survey sont affichées en pourcentage (`value × 100`), sans devenir des volumes et sans renormalisation à 100 %. Le graphique compare uniquement les marchés de la dernière année d'enquête disponible. Une somme inférieure à 100 % décrit le panel publié ; aucun marché résiduel n'est reconstruit.

In [7]:
prov_tanzania = provenance_df.loc[provenance_df.quality_flag.eq("survey_share_top15")].copy()
assert prov_tanzania.unit.eq("share").all()
assert prov_tanzania.metric_type.eq("source_market_share").all()
assert prov_tanzania.granularity.eq("country").all()
prov_tanzania["part_pct"] = 100 * prov_tanzania.value
prov_tz_coverage = prov_tanzania.groupby(["destination", "year", "coverage_scope"]).agg(
    marches=("origin_name", "nunique"), renseignes=("value", "count"),
    somme_parts_pct=("part_pct", lambda s: s.sum(min_count=1)))
display(prov_tz_coverage)
prov_tz_latest = prov_tanzania.loc[prov_tanzania.year.eq(prov_tanzania.year.max()) & prov_tanzania.value.notna()].copy()
assert prov_tz_latest[prov_dimensions].drop_duplicates().shape[0] == 1
prov_share_chart = alt.Chart(prov_tz_latest).mark_bar().encode(
    y=alt.Y("origin_name:N", title="Marché pays/territoire", sort="-x"),
    x=alt.X("part_pct:Q", title="Part publiée dans l'enquête (%)"),
    tooltip=["origin_name:N", "year:O", alt.Tooltip("part_pct:Q", format=".1f"), "coverage_scope:N"]
).properties(width=660, height=380, title=f"Tanzanie — Top 15 de l'Exit Survey, {int(prov_tz_latest.year.max())}")
display(prov_share_chart)
prov_tz_top = prov_tz_latest.sort_values("value", ascending=False).head(3)
display(Markdown("**Parts publiées, dernière enquête :** " + "; ".join(
    f"{r.origin_name} : {r.part_pct:.1f} %" for r in prov_tz_top.itertuples())
    + f". Somme des parts renseignées du panel : {prov_tz_latest.part_pct.sum(min_count=1):.1f} %, sans renormalisation."))

marches  \
destination year coverage_scope                                            
Tanzanie    2022 Top 15 marchés — parts issues de l'Exit Survey       15   
            2023 Top 15 marchés — parts issues de l'Exit Survey       15   
            2024 Top 15 marchés — parts issues de l'Exit Survey       15   

                                                                 renseignes  \
destination year coverage_scope                                               
Tanzanie    2022 Top 15 marchés — parts issues de l'Exit Survey          15   
            2023 Top 15 marchés — parts issues de l'Exit Survey          15   
            2024 Top 15 marchés — parts issues de l'Exit Survey          15   

                                                                 somme_parts_pct  
destination year coverage_scope                                                   
Tanzanie    2022 Top 15 marchés — parts issues de l'Exit Survey             80.1  
            2023 Top 15 marchés — parts issues de l'Exit Survey             77.5  
            2024 Top 15 marchés — parts issues de l'Exit Survey             81.9

alt.Chart(...)

**Parts publiées, dernière enquête :** United States : 15.0 %; Italy : 11.8 %; Kenya : 8.8 %. Somme des parts renseignées du panel : 81.9 %, sans renormalisation.

### 8.5 Contrôles et limites de comparabilité

Les classements ne comparent jamais des destinations, années, unités, granularités, périmètres ou types de mesure différents. Les Top-N et panels publiés ne garantissent pas l'exhaustivité nationale ; une origine non publiée n'est pas une origine de volume nul. Les libellés et compositions peuvent varier selon la source et l'année : aucune évolution d'un panel supposé constant n'est estimée ici. Les parts d'enquête tanzaniennes restent distinctes des volumes administratifs et des parts régionales égyptiennes.

In [8]:
pd.testing.assert_frame_equal(provenance_df, prov_initial, check_exact=True)
pd.testing.assert_frame_equal(df, prov_df_initial, check_exact=True)
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
assert prov_coverage.lignes.sum() == len(provenance_df)
assert prov_coverage.absentes.sum() == provenance_df.value.isna().sum()
assert provenance_df.loc[provenance_df.value.isna(), "quality_flag"].str.startswith("missing").all()
assert not provenance_df.loc[provenance_df.value.notna(), "quality_flag"].str.startswith("missing").any()
assert np.isfinite(provenance_df.value.dropna()).all()
assert provenance_df.value.dropna().ge(0).all()
assert provenance_df.loc[provenance_df.unit.eq("share"), "value"].dropna().between(0, 1).all()
assert prov_top.granularity.eq("country").all() and prov_top.unit.eq("persons").all()
assert prov_top.quality_flag.isin(prov_volume_flags).all()
assert not prov_top.quality_flag.eq("exact_single_country").any()
assert prov_top.index.isin(provenance_df.index).all()
pd.testing.assert_series_equal(prov_top.value, provenance_df.loc[prov_top.index, "value"])
np.testing.assert_allclose(prov_tanzania.part_pct, 100 * prov_tanzania.value, rtol=0, atol=0)
prov_chart_spec = prov_share_chart.to_dict(validate=True)
assert prov_chart_spec.get("datasets") and all(prov_chart_spec["datasets"].values())
print("Section 08 : contrôles réussis ; un graphique comparable, aucune reconstruction, sources intactes.")

Section 08 : contrôles réussis ; un graphique comparable, aucune reconstruction, sources intactes.


## 09 — Synthèse comparative

Les niveaux nationaux sont ceux de **2019**, année commune vérifiée en section 06. La dynamique est résumée par la médiane des variations annuelles avant 2020 ; la volatilité par leur écart-type échantillonnal, en points de pourcentage, sur la période commune calculée en section 07. Ces deux mesures ne sont ni des niveaux ni des CAGR.

La provenance reprend les résultats de la section 08 avec leur année, unité et périmètre propres. Elle est **non comparable entre les sept destinations**. Aucun score, classement composite ou graphique supplémentaire n'est nécessaire.

In [16]:
synthesis_inputs = {
    "cross_comparison": cross_comparison, "growth_summary": growth_summary,
    "prov_latest": prov_latest, "prov_tz_latest": prov_tz_latest, "prov_egypt": prov_egypt,
}
synthesis_snapshots = {k: v.copy(deep=True) for k, v in synthesis_inputs.items()}
assert cross_last_year == 2019, "La synthèse exige une comparaison nationale commune en 2019."
assert 2019 in cross_common_years
synthesis_table = cross_comparison.set_index("destination")[["arrivals", "receipts", RATIO]].reindex(DESTINATIONS).copy()
synthesis_table.columns = ["Arrivées 2019 (personnes)", "Recettes 2019 (USD courants)", "Ratio 2019 (USD courants/arrivée)"]
for synthesis_label in ["Arrivées", "Recettes"]:
    synthesis_stats = growth_summary.loc[synthesis_label].reindex(DESTINATIONS)
    synthesis_table[f"{synthesis_label} : variation annuelle médiane (%)"] = synthesis_stats["mediane_pct"]
    synthesis_table[f"{synthesis_label} : volatilité (points)"] = synthesis_stats["volatilite_points"]
synthesis_table["Période dynamique/volatilité"] = (
    f"{min(growth_common_years)}–{max(growth_common_years)} ; {len(growth_common_years)} variations annuelles")
synthesis_table["Provenance : constat disponible"] = "Données insuffisantes"
synthesis_table["Provenance : périmètre et comparabilité"] = "Non comparable entre destinations"
for synthesis_destination in DESTINATIONS:
    synthesis_market = prov_latest.loc[prov_latest.destination.eq(synthesis_destination)]
    synthesis_share = prov_tz_latest.loc[prov_tz_latest.destination.eq(synthesis_destination)]
    synthesis_egypt = prov_egypt.loc[prov_egypt.destination.eq(synthesis_destination)]
    if not synthesis_market.empty:
        synthesis_leaders = synthesis_market.loc[synthesis_market.rang_dans_perimetre.eq(1)]
        synthesis_table.loc[synthesis_destination, "Provenance : constat disponible"] = "; ".join(
            f"{r.year} : {r.origin_name}, {r.value:,.0f} personnes (premier parmi les pays/territoires publiés)"
            for r in synthesis_leaders.itertuples())
        synthesis_scope = "; ".join(synthesis_market.coverage_scope.unique())
        synthesis_table.loc[synthesis_destination, "Provenance : périmètre et comparabilité"] = (
            synthesis_scope + " ; non comparable entre destinations ; exhaustivité nationale non présumée")
    elif not synthesis_share.empty:
        synthesis_leaders = synthesis_share.loc[synthesis_share.value.eq(synthesis_share.value.max())]
        synthesis_table.loc[synthesis_destination, "Provenance : constat disponible"] = "; ".join(
            f"{r.year} : {r.origin_name}, {r.part_pct:.1f} % (part d'enquête)"
            for r in synthesis_leaders.itertuples())
        synthesis_table.loc[synthesis_destination, "Provenance : périmètre et comparabilité"] = (
            "; ".join(synthesis_share.coverage_scope.unique()) + " ; non comparable aux volumes ; panel partiel")
    elif not synthesis_egypt.empty:
        synthesis_table.loc[synthesis_destination, "Provenance : constat disponible"] = (
            f"{synthesis_egypt.loc[synthesis_egypt.granularity.eq('country'), 'origin_name'].nunique()} marché pays vérifié ; "
            "données insuffisantes pour identifier les principaux marchés")
        synthesis_table.loc[synthesis_destination, "Provenance : périmètre et comparabilité"] = (
            "Marché pays isolé et parts régionales touristes/nuitées distinctes ; non comparable")
display(synthesis_table.style.format({
    c: "{:,.0f}" if c in ["Arrivées 2019 (personnes)", "Recettes 2019 (USD courants)"] else "{:,.2f}"
    for c in synthesis_table.select_dtypes(include="number").columns
}, na_rep="Données insuffisantes"))


,Arrivées 2019 (personnes),Recettes 2019 (USD courants),Ratio 2019 (USD courants/arrivée),Arrivées : variation annuelle médiane (%),Arrivées : volatilité (points),Recettes : variation annuelle médiane (%),Recettes : volatilité (points),Période dynamique/volatilité,Provenance : constat disponible,Provenance : périmètre et comparabilité
destination,,,,,,,,,,
Afrique du Sud,"14,797,000","9,064,000,000",612.56,4.15,6.15,0.50,18.78,1998–2019 ; 22 variations annuelles,"2024 : Zimbabwe, 2,183,260 personnes (premier parmi les pays/territoires publiés)",Panel harmonisé: 10 marchés SADC + 8 marchés overseas ; non comparable entre destinations ; exhaustivité nationale non présumée
Égypte,"13,026,000","14,256,000,000","1,094.43",13.24,23.23,13.34,42.12,1998–2019 ; 22 variations annuelles,1 marché pays vérifié ; données insuffisantes pour identifier les principaux marchés,Marché pays isolé et parts régionales touristes/nuitées distinctes ; non comparable
Kenya,"2,049,000","1,762,000,000",859.93,8.06,15.32,6.73,21.03,1998–2019 ; 22 variations annuelles,"2024 : United States of America, 306,501 personnes (premier parmi les pays/territoires publiés)",Top 30 marchés sources publiés ; non comparable entre destinations ; exhaustivité nationale non présumée
Maroc,"13,109,000","9,949,000,000",758.94,5.76,5.96,8.78,11.65,1998–2019 ; 22 variations annuelles,"2020 : France, 412,179 personnes (premier parmi les pays/territoires publiés)",Nationalités/pays publiés par Open Data Maroc ; non comparable entre destinations ; exhaustivité nationale non présumée
Maurice,"1,418,000","2,024,000,000","1,427.36",3.86,5.08,7.85,10.92,1998–2019 ; 22 variations annuelles,"2024 : France, 339,421 personnes (premier parmi les pays/territoires publiés)",7 principaux marchés publiés dans les annual highlights ; non comparable entre destinations ; exhaustivité nationale non présumée
Tanzanie,"1,527,000","2,624,500,000","1,718.73",5.10,11.62,8.48,15.71,1998–2019 ; 22 variations annuelles,"2024 : United States, 15.0 % (part d'enquête)",Top 15 marchés — parts issues de l'Exit Survey ; non comparable aux volumes ; panel partiel
Tunisie,"9,429,000","2,683,000,000",284.55,4.86,12.35,5.97,15.94,1998–2019 ; 22 variations annuelles,"2023 : Algériens, 3,038,002 personnes (premier parmi les pays/territoires publiés)",Nationalités publiées par ONTT ; non comparable entre destinations ; exhaustivité nationale non présumée


### Lecture comparative et limites

Les niveaux et les variations nominales des recettes restent sensibles aux prix et aux changes : aucune correction d'inflation n'est disponible. Le ratio agrégé n'est ni une dépense individuelle, ni une rentabilité, ni une mesure de qualité. La volatilité décrit la dispersion passée et n'établit aucune causalité. L'année 2020 reste une rupture traitée séparément en section 07 ; les données nationales ne permettent pas de mesurer une reprise post-Covid. Les années récentes de provenance ne comblent pas cette lacune.

In [17]:
for synthesis_field, synthesis_description in [
    ("arrivals", "arrivées"), ("receipts", "recettes"), (RATIO, "ratio agrégé recettes/arrivées")]:
    synthesis_maximum = cross_comparison[synthesis_field].max()
    synthesis_names = ", ".join(cross_comparison.loc[cross_comparison[synthesis_field].eq(synthesis_maximum), "destination"])
    display(Markdown(f"**Niveaux en {cross_last_year} :** {synthesis_names} présente le maximum de {synthesis_description} "
        f"({synthesis_maximum:,.2f}, dans l'unité indiquée au tableau). Ces maxima ne définissent pas une performance globale."))
for synthesis_label in ["Arrivées", "Recettes"]:
    synthesis_stats = growth_summary.loc[synthesis_label]
    for synthesis_field, synthesis_description in [
        ("mediane_pct", "variation annuelle médiane la plus élevée (%)"),
        ("volatilite_points", "dispersion annuelle la plus élevée (points)")]:
        synthesis_maximum = synthesis_stats[synthesis_field].max()
        synthesis_names = ", ".join(synthesis_stats.index[synthesis_stats[synthesis_field].eq(synthesis_maximum)])
        display(Markdown(f"**{synthesis_label}, {min(growth_common_years)}–{max(growth_common_years)} :** "
            f"{synthesis_names} présente la {synthesis_description} : {synthesis_maximum:.2f}."))
for synthesis_name, synthesis_input in synthesis_inputs.items():
    pd.testing.assert_frame_equal(synthesis_input, synthesis_snapshots[synthesis_name], check_exact=True)
assert len(synthesis_table) == len(DESTINATIONS) == 7
assert synthesis_table.index.is_unique and synthesis_table.index.tolist() == DESTINATIONS
assert synthesis_table.iloc[:, :7].notna().all().all()
np.testing.assert_array_equal(synthesis_table.iloc[:, :3].to_numpy(),
    cross_comparison.set_index("destination").reindex(DESTINATIONS)[["arrivals", "receipts", RATIO]].to_numpy())
for synthesis_label in ["Arrivées", "Recettes"]:
    for synthesis_column, synthesis_source_column in [
        (f"{synthesis_label} : variation annuelle médiane (%)", "mediane_pct"),
        (f"{synthesis_label} : volatilité (points)", "volatilite_points")]:
        np.testing.assert_array_equal(synthesis_table[synthesis_column],
            growth_summary.loc[synthesis_label].reindex(DESTINATIONS)[synthesis_source_column])
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == source_hash
pd.testing.assert_frame_equal(df, df_initial, check_exact=True)
print("Section 09 validée : sept destinations, résultats repris à l'identique, aucune pondération ni reconstruction.")


**Niveaux en 2019 :** Afrique du Sud présente le maximum de arrivées (14,797,000.00, dans l'unité indiquée au tableau). Ces maxima ne définissent pas une performance globale.

**Niveaux en 2019 :** Égypte présente le maximum de recettes (14,256,000,000.00, dans l'unité indiquée au tableau). Ces maxima ne définissent pas une performance globale.

**Niveaux en 2019 :** Tanzanie présente le maximum de ratio agrégé recettes/arrivées (1,718.73, dans l'unité indiquée au tableau). Ces maxima ne définissent pas une performance globale.

**Arrivées, 1998–2019 :** Égypte présente la variation annuelle médiane la plus élevée (%) : 13.24.

**Arrivées, 1998–2019 :** Égypte présente la dispersion annuelle la plus élevée (points) : 23.23.

**Recettes, 1998–2019 :** Égypte présente la variation annuelle médiane la plus élevée (%) : 13.34.

**Recettes, 1998–2019 :** Égypte présente la dispersion annuelle la plus élevée (points) : 42.12.

Section 09 validée : sept destinations, résultats repris à l'identique, aucune pondération ni reconstruction.


## 10 — Limites méthodologiques

### Couverture temporelle et valeurs manquantes

Les périodes disponibles varient selon les destinations et les mesures. Les comparaisons nationales doivent porter sur des années communes renseignées ; les résultats sur des périodes propres ne sont pas directement comparables. Les séries nationales s'arrêtent au plus tard en 2020 : elles ne permettent pas d'analyser fiablement la reprise post-Covid. Les données de provenance plus récentes ne comblent pas cette absence.

**Une donnée manquante n'est pas un zéro.** Aucune interpolation, extrapolation ou reconstruction n'est admise. Les variations annuelles nécessitent deux années consécutives renseignées et une base strictement positive ; les trous limitent la couverture des analyses.

### Unités et portée des indicateurs

Les recettes sont exprimées en **USD courants**, sans correction de l'inflation. Leur évolution nominale peut notamment refléter les prix et les changes ; elle ne mesure pas une croissance réelle.

Le **ratio recettes/arrivées est un ratio agrégé** de séries dont les définitions peuvent différer. Il ne mesure ni une dépense individuelle, ni une rentabilité, ni la qualité touristique. Un ratio élevé ne suffit donc pas à conclure à une meilleure performance.

Les variations, la volatilité et les corrélations sont **descriptives, sans portée causale**. Une corrélation groupant destinations et années peut refléter des différences structurelles ou des tendances partagées. La volatilité passée ne prédit pas un risque futur ; 2020 reste une rupture exceptionnelle traitée séparément.

### Comparabilité des provenances

Les sources diffèrent par leurs années, définitions, Top-N et panels partiels. Pays/territoires, agrégats régionaux, diasporas et catégories institutionnelles ne constituent pas des catégories interchangeables. Les totaux ne doivent pas être additionnés à leurs composantes ; volumes et parts, ainsi que parts de touristes et de nuitées, restent distincts.

Les principaux marchés identifiés le sont **uniquement dans le périmètre publié** : aucun classement global entre les sept destinations ni exhaustivité nationale ne peut en être déduit. Une origine non publiée ne représente pas un volume nul.

- **Tanzanie :** les parts publiées de l'enquête restent des parts, sans conversion en volumes, renormalisation ou reconstruction d'un marché résiduel.
- **Égypte :** la couverture des marchés pays est insuffisante pour un classement complet. Le marché pays vérifié et les répartitions régionales ne permettent pas de reconstituer les marchés absents.

### Définitions et traçabilité à confirmer

Les réserves déjà signalées dans les métadonnées du dataset restent applicables. Il faut conserver et consulter `granularity`, `metric_type`, `coverage_scope`, `quality_flag` et les références/notes de source. Lorsqu'une définition ou une valeur reste à vérifier, elle ne doit pas être présentée comme validée ni servir à une interprétation économique avancée sans vérification complémentaire.


## 11 — Insights principaux

Ces six constats reprennent les résultats validés des sections 04–10, sans nouveau calcul. Ils décrivent les données disponibles et ne constituent ni un score global ni une explication causale.

1. **Trois indicateurs, trois leaders en 2019.** Sur l'année nationale commune, l'Afrique du Sud compte le plus d'arrivées (**14 797 000**), l'Égypte les recettes les plus élevées (**14,256 milliards USD courants**) et la Tanzanie le ratio recettes/arrivées le plus élevé (**1 718,73 USD courants par arrivée**). Le futur dashboard doit présenter ces indicateurs séparément : le ratio agrégé ne mesure ni dépense individuelle, ni rentabilité, ni qualité touristique. *Références : sections 06 et 09.*

2. **La dynamique médiane avant 2020 est la plus élevée en Égypte.** Sur les **22 variations annuelles communes de 1998 à 2019**, l'Égypte présente les médianes maximales pour les arrivées (**13,24 %**) et les recettes (**13,34 %**). Ces médianes de variations annuelles ne sont pas des CAGR ; les recettes restent nominales. Le dashboard doit distinguer dynamique et niveau atteint. *Références : sections 07 et 09.*

3. **La dispersion des variations oppose notamment Égypte et Maurice.** Sur cette même période commune, la volatilité est maximale en Égypte : **23,23 points** pour les arrivées et **42,12 points** pour les recettes. Elle est minimale à Maurice : **5,08** et **10,92 points**, respectivement. Cet écart-type décrit l'irrégularité passée des variations, sans prédire un risque futur ni expliquer ses causes. Une lecture de la croissance doit donc conserver l'information de dispersion. *Référence : section 07.*

4. **2020 constitue une rupture à présenter séparément.** Les variations d'arrivées calculables pour **4 destinations** vont de **−78,66 % à −73,73 %** ; celles des recettes, calculables pour **5 destinations**, de **−74,41 % à −54,63 %**. Ces intervalles portent sur des couvertures différentes et ne décrivent pas les sept destinations. Les **3 variations d'arrivées et 2 variations de recettes absentes** doivent rester signalées comme non calculables, jamais comme nulles. *Références : sections 07 et 10.*

5. **La reprise nationale post-Covid n'est pas mesurable dans ce dataset.** Les résultats recensent **0 observation nationale renseignée après 2020**. Le dashboard ne peut donc afficher un taux de reprise nationale fiable ; les provenances plus récentes ne remplacent pas les séries nationales manquantes. *Références : sections 07 et 10.*

6. **La provenance exige des vues propres à chaque périmètre.** Les **716 lignes** comprennent **656 valeurs renseignées et 60 absences**. Les panels publiés, pays/territoires, agrégats régionaux, diasporas et catégories institutionnelles ne permettent pas un classement global des destinations. En Tanzanie, les parts d'enquête de 2024 placent les États-Unis à **15 %**, l'Italie à **11,8 %** et le Kenya à **8,8 %** ; les parts du panel totalisent **81,9 %**, sans conversion en volumes ni renormalisation. En Égypte, **un seul marché pays est vérifié** : un classement complet serait injustifié. Toute vue de provenance doit afficher année, unité, couverture et qualité. *Références : sections 08–10.*


## 12 — Indicateurs recommandés pour le dashboard

Cette sélection reprend les résultats et limites des sections 04–11. Elle ne crée ni nouvel indicateur composite, ni données, ni graphique dans le notebook. Toute valeur affichée doit conserver son année, son unité et son périmètre ; une absence est affichée « Non disponible » ou « Non calculable », jamais remplacée par zéro.

### 1. Tendances nationales

| Indicateur et définition | Unité | Filtres pertinents | Visualisation recommandée | Limite / interprétation à afficher |
|---|---|---|---|---|
| **Arrivées touristiques** : valeur publiée de la couche nationale `arrivals`, de type `destination_total`. | Personnes, selon la série source | Destination, période/année, couche et granularité nationales ; valeurs renseignées | Courbe annuelle avec interruptions aux années absentes ; carte de valeur pour une année explicitement choisie | Couverture propre à chaque destination ; les données nationales s'arrêtent au plus tard en 2020. Ne pas présenter une dernière valeur disponible comme une valeur actuelle. |
| **Recettes touristiques** : valeur publiée de la couche nationale `receipts`, de type `destination_total`. | USD courants ; milliards USD courants possibles pour l'affichage | Destination, période/année, couche et granularité nationales ; valeurs renseignées | Courbe annuelle séparée de celle des arrivées ; carte de valeur datée | Montants nominaux, sans correction d'inflation ; les variations ne mesurent pas une croissance réelle. |
| **Variation annuelle des arrivées ou des recettes** : `100 × (valeur_t / valeur_t−1 − 1)`, pour un même indicateur et une même destination. | % | Indicateur, destination, année de fin ; deux années consécutives renseignées, base strictement positive | Barres divergentes autour de zéro ou carte thermique ; 2020 présenté séparément de la période antérieure | Aucun calcul entre années non consécutives, aucune interpolation ; « Non calculable » si une condition manque. Variation descriptive, nominale pour les recettes. |
| **Ratio recettes touristiques / arrivées** : quotient des deux valeurs nationales de la même destination et de la même année. | USD courants par arrivée | Destination, année/période ; couple complet, arrivées strictement positives, unités et définitions vérifiées | Courbe segmentée aux absences ; carte de valeur datée ; barres sur une année commune | Ratio agrégé, sans correction d'inflation : ni dépense individuelle, ni rentabilité, ni qualité touristique. |

### 2. Comparaison des destinations

| Indicateur et définition | Unité | Filtres pertinents | Visualisation recommandée | Limite / interprétation à afficher |
|---|---|---|---|---|
| **Niveaux nationaux sur une année commune** : arrivées, recettes et ratio déjà définis ci-dessus, pour les mêmes destinations et la même année. | Trois unités distinctes : personnes ; USD courants ; USD courants par arrivée | Ensemble de destinations, indicateur, année complète commune ; **2019** pour la comparaison validée des sept destinations | Barres distinctes par indicateur ou tableau ; nuage arrivées × recettes en complément, avec destination et année en infobulle | Aucun mélange de dernières années propres. Si l'année n'est pas commune, signaler « Non comparable ». Les leaders peuvent différer selon l'indicateur ; aucun score global. |
| **Dynamique avant 2020** : médiane des variations annuelles validées, séparément pour les arrivées et les recettes. | % | Destination, indicateur, même ensemble d'années de variation ; fenêtre validée **1998–2019**, 22 variations par série | Tableau ou points comparatifs, avec période et effectif | Médiane de variations annuelles, pas un CAGR ni une variation cumulée. Recettes nominales. Toute autre fenêtre exige une vérification préalable de la couverture commune. |
| **Volatilité avant 2020** : écart-type échantillonnal des variations annuelles, séparément pour chaque indicateur. | Points de pourcentage | Mêmes filtres et années communes que pour la dynamique ; au moins deux variations valides | Points ou barres dans une vue distincte de la dynamique ; période et effectif visibles | Dispersion passée, sans causalité ni prédiction de risque. Ne pas confondre forte croissance et forte volatilité ; ne pas incorporer 2020 à la fenêtre pré-2020. |

Les niveaux, la dynamique et la volatilité restent trois dimensions distinctes, sans pondération ni classement composite.

### 3. Provenance des visiteurs

Toutes les vues affichent **destination, année, type de mesure, unité, granularité, périmètre/couverture et qualité**, avec accès aux références et notes de source. Les filtres respectent `granularity`, `metric_type`, `coverage_scope` et `quality_flag`. Une comparaison entre destinations n'est activée qu'après vérification de définitions, années, unités et périmètres réellement compatibles ; la couche actuelle ne justifie aucun classement global des sept destinations.

| Indicateur et définition | Unité | Filtres pertinents | Visualisation recommandée | Limite / interprétation à afficher |
|---|---|---|---|---|
| **Volumes par marché pays/territoire** : valeurs publiées de type `tourist_arrivals` et de granularité `country`. | Personnes | Destination, année, pays/territoire, périmètre et qualité compatibles ; valeurs renseignées | Barres des principaux marchés au sein du même périmètre ; tableau précisant le nombre de marchés publiés | Rang interne au panel publié, sans exhaustivité nationale présumée. Top-N et panels partiels explicités ; origine absente ≠ volume nul. En Égypte, données insuffisantes pour identifier les principaux marchés pays. |
| **Parts des marchés tanzaniens** : parts publiées de type `source_market_share` dans l'Exit Survey. | % : affichage de `100 × value` | Tanzanie, année d'enquête, granularité pays, périmètre Top 15, qualité `survey_share_top15` | Barres des parts publiées pour une seule année | Parts d'enquête, pas des volumes. Aucune conversion en personnes, renormalisation à 100 % ou reconstruction de catégories résiduelles. |
| **Parts régionales égyptiennes** : répartitions publiées, séparément pour `regional_tourist_share` et `regional_tourist_nights_share`. | % | Égypte, année disponible, type de part, agrégat régional, périmètre et qualité | Deux panneaux distincts ou tableau séparant touristes et nuitées | Touristes et nuitées ne sont pas interchangeables. Couverture régionale limitée ; aucune déduction de volumes ou de classement pays. |
| **Agrégats, diasporas et catégories institutionnelles publiés** : valeurs conservées sous leur granularité et leur type propres. | Unité publiée, sans mélange | Destination, année, granularité, type de mesure, couverture et qualité | Tableau par catégorie ; barres uniquement au sein d'un groupe compatible et sans chevauchement établi | Ne pas mélanger pays, régions, MRE/TRE, institutions et totaux. Ne pas additionner agrégats et composantes. Ces catégories ne constituent pas des marchés pays. |

### Indicateurs à ne pas présenter comme KPI

- **Reprise nationale post-Covid** : absence de séries nationales renseignées après 2020 ; les provenances récentes ne les remplacent pas.
- **Dépense moyenne par touriste** : le ratio de séries agrégées ne démontre pas une mesure individuelle.
- **Rentabilité ou qualité touristique déduite du ratio** : absence des données nécessaires à ces interprétations.
- **Classement global des provenances** : années, panels, granularités et mesures hétérogènes.
- **Score composite arbitraire** : aucune pondération entre niveaux, dynamique, volatilité ou provenance n'est justifiée par l'EDA.
